<a href="https://colab.research.google.com/github/farrelrassya/Build-a-Large-Language-Model-From-Scratch/blob/main/03.Coding_Attention_Mechanisms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farrelrassya/Build-a-Large-Language-Model-From-Scratch/blob/main/03.Coding_Attention_Mechanisms.ipynb)

# Chapter 3 -- Coding Attention Mechanisms

*Companion notebook untuk* Build a Large Language Model (From Scratch) *(Sebastian Raschka, Manning 2024)*
**Technical edition -- kode buku plus beberapa pengukuran tambahan**

Di Chapter 2, teks sudah berubah menjadi deretan embedding: satu vektor untuk setiap token. Tapi setiap vektor itu masih berdiri sendiri. Embedding token untuk "journey" selalu sama, apa pun kalimatnya. Embedding posisi dari Chapter 2 memang menambahkan informasi tentang letak kata itu, tapi tidak satu pun dari keduanya tahu kata apa yang ada di sebelahnya. Padahal arti sebuah kata sering baru jelas dari kata-kata di sekitarnya.

Bab ini membangun mekanisme yang mengisi kekurangan itu, yaitu *attention*. Gagasannya bisa diucapkan dalam satu kalimat: setiap token membuat versi baru dirinya, berupa rata-rata berbobot dari semua token di kalimat yang sama, dengan bobot besar untuk token yang paling relevan. Hampir seluruh bab adalah jawaban atas satu pertanyaan: dari mana bobot itu datang?

Jawabannya dibangun dalam empat tahap, dan setiap tahap menambal kekurangan tahap sebelumnya. Mula-mula bobot dihitung langsung dari embedding, tanpa apa pun yang bisa dilatih. Lalu ditambahkan tiga matriks yang bisa dilatih, sehingga model bisa belajar sendiri apa arti "relevan". Sesudah itu setiap token dilarang melihat token sesudahnya, syarat mutlak bagi model yang menulis teks dari kiri ke kanan. Terakhir, beberapa attention dijalankan berdampingan sebagai *multi-head attention*, modul yang nanti dipasang di dalam GPT pada Chapter 4. Satu kalimat pendek, "Your journey starts with one step", menemani kita sepanjang bab, dan token "journey" hampir selalu menjadi token yang kita ikuti.

Di beberapa titik ada pengukuran tambahan yang tidak ada di buku, untuk menguji klaim yang di buku hanya dijelaskan dengan kata-kata. Bagian-bagian itu berjudul **Under the hood** dan boleh dilewati kalau hanya ingin mengikuti materi buku.

| Under the hood | Yang kita ukur | Hasil |
|---|---|---|
| 1. Kenapa skor dibagi $\sqrt{d_k}$ | Bentuk softmax saat dimensi query dan key membesar, dengan dan tanpa pembagi | Tanpa pembagi, di $d_k = 4096$ bobot terbesar rata-rata 0.986, nyaris semuanya ke satu token; dengan pembagi, antara 0.38 dan 0.43 di semua ukuran |
| 2. Causal mask | Apakah token sesudahnya benar-benar tidak memengaruhi token sebelumnya | Mengganggu token 4-6 tidak mengubah satu bit pun output token 1-3 |
| 3. Dropout | Apa yang dilakukan dropout pada bobot attention, dan kapan ia benar-benar mati | Satu baris bobot bisa berjumlah 0 atau 2 saat training; `no_grad()` tidak mematikan dropout, hanya `eval()` |
| 4. Wrapper vs `MultiHeadAttention` | Apakah dua implementasi multi-head menghitung fungsi yang sama, dan berapa biayanya | Sama sampai pembulatan float32; tapi di CPU satu thread pada run ini, versi yang disebut lebih efisien justru lebih lambat |

Di akhir bab kita juga memeriksa ukuran GPT-2 yang disebut buku. Model terkecilnya ternyata berisi 124 juta parameter, bukan 117 juta.

## Setup

Kode setiap listing diambil dari repo resmi buku, `rasbt/LLMs-from-scratch` (commit `cdbd33e`, lisensi Apache-2.0), dan disusun mengikuti urutan buku. Di beberapa tempat versi repo sedikit berbeda dari versi cetak, misalnya pada cara menulis argumen softmax; perbedaan itu kita bahas saat bertemu. Sel yang tidak berasal dari buku maupun repo kita sebut *sel tambahan*.

Dua sel berikut mencatat lingkungan run. Yang pertama adalah sel versi dari repo, yang kedua sel tambahan yang mencatat mesin dan memilih `device`.

In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.11.0+cu130


Run ini memakai PyTorch 2.11.0; akhiran `+cu130` menandakan build dengan dukungan CUDA 13.0. Output yang tersimpan di repo untuk sel yang sama mencetak `2.4.0`. Selisih versi sebesar itu tidak terlihat pengaruhnya di bab ini: di setiap tempat kita mencocokkan angka listing dengan buku, hasilnya sama. Untuk mask dropout, pencocokan itu bahkan dikerjakan otomatis di Under the hood 3.

In [2]:
import os
import platform
import subprocess
import torch

device = torch.device(os.environ.get("NB_DEVICE") or
                      ("cuda" if torch.cuda.is_available() else
                       "mps" if torch.backends.mps.is_available() else "cpu"))

cpu_name = platform.processor()
if platform.system() == "Darwin":
    cpu_name = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"],
                              capture_output=True, text=True).stdout.strip()
HOST_LABEL = f"{cpu_name}, {platform.system()} {platform.release()} {platform.machine()}"

print(f"host    : {HOST_LABEL}")
print(f"python  : {platform.python_version()} | torch {torch.__version__}")
print(f"backend : cuda={torch.cuda.is_available()} mps={torch.backends.mps.is_available()} "
      f"MKL={torch.backends.mkl.is_available()} threads={torch.get_num_threads()}")
print(f"device  : {device}" + ("  (override NB_DEVICE)" if os.environ.get("NB_DEVICE") else ""))


host    : x86_64, Linux 6.6.122+ x86_64
python  : 3.13.15 | torch 2.11.0+cu130
backend : cuda=True mps=False MKL=True threads=1
device  : cuda


Run ini berjalan di Linux x86_64 dengan Python 3.13.15, dan GPU CUDA tersedia (`cuda=True`), sehingga `device` otomatis menjadi `cuda`. Urutan pilihannya CUDA, lalu MPS (GPU Apple), lalu CPU; variabel lingkungan `NB_DEVICE` bisa memaksa pilihan lain.

Tapi angka-angka bab ini tidak dihitung di GPU. Listing buku membuat semua tensor di CPU, bawaan PyTorch, dan tidak pernah memindahkannya ke `device`. Variabel itu hanya dipakai di satu tempat, yaitu Under the hood 3, untuk membandingkan mask dropout dari CPU dan dari GPU. Satu keterangan lain di baris `backend` perlu diingat: `threads=1`, artinya operasi PyTorch di CPU berjalan dengan satu thread saja. Angka ini akan penting ketika kita mengukur kecepatan di Under the hood 4.

Bab ini dibuka dengan peta jalan: di mana attention berada dalam proses membangun LLM (figure 3.1), dan empat varian attention yang akan kita bangun berurutan (figure 3.2).

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.1.png" width="700">

**Figure 3.1** Tiga tahap membangun LLM. Bab ini mengerjakan langkah 2 di tahap pertama: mekanisme attention, komponen inti di dalam arsitektur LLM.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.2.png" width="700">

**Figure 3.2** Empat varian attention yang dibangun berurutan: self-attention sederhana, self-attention dengan bobot yang bisa dilatih, causal attention, dan multi-head attention. Setiap varian menambahkan satu hal pada varian sebelumnya.

## 3.1 The problem with modeling long sequences

Sebelum masuk ke attention, ada baiknya kita lihat dulu masalah yang mendorong lahirnya mekanisme ini. Ambil contoh penerjemahan. Menerjemahkan kata per kata hampir selalu gagal, karena setiap bahasa menyusun kalimat dengan caranya sendiri. Dalam bahasa Jerman, misalnya, kata kerja bisa muncul di ujung kalimat, padahal dalam bahasa Inggris kata kerja itu muncul jauh lebih awal (figure 3.3). Penerjemah harus membaca seluruh kalimat sumber dulu, baru menulis.

Sebelum ada transformer, cara standar untuk itu adalah *encoder-decoder RNN* (recurrent neural network). Encoder membaca kalimat sumber token demi token sambil terus memperbarui satu *hidden state*, semacam ringkasan berjalan dari semua yang sudah dibacanya. Sesudah token terakhir, decoder mulai menulis terjemahan, juga token demi token, berangkat dari hidden state terakhir itu (figure 3.4).

Kelemahannya ada di pemadatan itu. Saat menulis, decoder hanya memegang hidden state terakhir; hidden state yang lebih awal sudah tidak bisa diaksesnya. Seluruh makna kalimat harus muat di satu vektor, dan pada kalimat panjang, hubungan antara kata-kata yang berjauhan mudah hilang di perjalanan.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.3.png" width="700">

**Figure 3.3** Menerjemahkan dari bahasa Jerman ke bahasa Inggris kata per kata menghasilkan kalimat yang salah tata bahasanya. Terjemahan yang benar perlu melihat kata yang muncul lebih awal atau lebih akhir di kalimat sumber.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.4.png" width="700">

**Figure 3.4** Encoder-decoder RNN. Encoder memadatkan seluruh input ke dalam hidden state, lalu decoder menghasilkan terjemahan token demi token dari state tersebut.

## 3.2 Capturing data dependencies with attention mechanisms

Jalan keluarnya muncul pada 2014, lewat mekanisme yang sekarang dikenal sebagai *Bahdanau attention*. Encoder-decoder RNN-nya tetap, tapi decoder kini boleh menengok ke semua posisi input di setiap langkah. Setiap kali decoder menulis satu token, setiap token input diberi bobot: seberapa penting ia untuk token yang sedang ditulis (figure 3.5). Decoder tidak lagi bergantung pada satu ringkasan; ia memilih sendiri bagian input mana yang perlu dilihat.

Tiga tahun kemudian, arsitektur transformer menunjukkan bahwa RNN-nya sendiri tidak diperlukan. Yang dipertahankan hanya gagasan pembobotan itu, dalam bentuk *self-attention*: setiap posisi dalam satu sekuens menimbang semua posisi lain di sekuens yang sama. Figure 3.6 memperlihatkan letak modul ini di dalam GPT. Bab ini membangun modul itu sendiri; bagian arsitektur di sekitarnya menyusul di Chapter 4.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.5.png" width="700">

**Figure 3.5** Dengan attention, decoder bisa mengakses semua token input secara selektif. Tebal garis menggambarkan seberapa penting sebuah token input untuk token output yang sedang dihasilkan.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.6.png" width="700">

**Figure 3.6** Self-attention adalah salah satu modul di dalam GPT, sebuah model decoder-only. Persiapan data (Chapter 2) berada sebelum modul ini, dan sisa arsitekturnya (Chapter 4) berada di sekitarnya.

## 3.3 Attending to different parts of the input with self-attention

### 3.3.1 A simple self-attention mechanism without trainable weights

Kata "self" di sini penting. Attention Bahdanau menghubungkan dua sekuens, kalimat sumber dan terjemahannya. Self-attention bekerja di dalam *satu* sekuens: setiap token menimbang token-token lain di kalimat yang sama, termasuk dirinya sendiri.

Untuk setiap embedding input $x^{(i)}$, hasilnya adalah sebuah *context vector* $z^{(i)}$, yaitu rata-rata berbobot dari semua embedding input, dengan bobot yang ditentukan dari sudut pandang token ke-$i$. Bayangkan saja sebagai embedding token yang sudah "membaca" kalimatnya. Contoh berjalan kita adalah kalimat "Your journey starts with one step": enam token, masing-masing dengan embedding 3 dimensi supaya angkanya bisa diikuti dengan mata. Kita akan menghitung $z^{(2)}$, context vector untuk token kedua, "journey".

Varian pertama ini sengaja belum punya bobot yang bisa dilatih. Tujuannya memperkenalkan tiga langkah yang akan terus berulang di sepanjang bab: hitung skor, ubah skor menjadi bobot, lalu jumlahkan input menurut bobotnya.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.7.png" width="700">

**Figure 3.7** Context vector $z^{(2)}$ adalah gabungan semua input, masing-masing ditimbang menurut relevansinya terhadap $x^{(2)}$. Bobot $\alpha_{21} \ldots \alpha_{2T}$ menentukan seberapa besar sumbangan setiap input.

In [3]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

Satu baris untuk setiap token, satu kolom untuk setiap dimensi embedding. Angka-angka ini dibuat untuk contoh, bukan hasil training, tapi perhatikan baris kedua dan ketiga: "journey" `[0.55, 0.87, 0.66]` dan "starts" `[0.57, 0.85, 0.64]` hampir kembar. Kemiripan itu akan muncul lagi berkali-kali.

Langkah 1 adalah menghitung *attention score*, yaitu seberapa cocok setiap token dengan token yang sedang kita perhatikan. Token yang sedang diperhatikan disebut *query*, dan di sini query-nya "journey". Ukuran kecocokan yang paling sederhana adalah *dot product*: kalikan dua vektor elemen demi elemen, lalu jumlahkan hasilnya. Skor untuk pasangan "journey" dan token ke-$j$ ditulis $\omega_{2j} = x^{(2)} \cdot x^{(j)}$.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.8.png" width="700">

**Figure 3.8** Langkah 1: skor attention adalah dot product antara query $x^{(2)}$ dan setiap input. Angka di gambar ditampilkan dengan satu desimal.

In [4]:
query = inputs[1]  # 2nd input token is the query

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # dot product (transpose not necessary here since they are 1-dim vectors)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


Enam skor untuk query "journey": `[0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865]`, sama dengan angka di buku. Skor tertinggi, 1.4950, adalah skor "journey" dengan dirinya sendiri. Itu tidak mengejutkan, karena dot product sebuah vektor dengan dirinya sendiri adalah kuadrat panjangnya: $0.55^2 + 0.87^2 + 0.66^2 = 1.4950$. Peringkat kedua adalah "starts" dengan 1.4754, karena embedding-nya hampir kembar dengan "journey".

Tapi dot product mengukur dua hal sekaligus, arah dan panjang. Berbeda dengan *cosine similarity*, ia tidak dinormalisasi, sehingga vektor yang lebih panjang otomatis mendapat skor lebih besar. Akibatnya akan terlihat jelas di 3.3.2.

Sel berikut menghitung skor pertama secara manual, untuk memastikan tidak ada yang tersembunyi di balik `torch.dot`.

In [5]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


Hasilnya sama, 0.9544, dan bisa diperiksa dengan tangan: $0.43 \cdot 0.55 + 0.15 \cdot 0.87 + 0.89 \cdot 0.66 = 0.2365 + 0.1305 + 0.5874$. Dot product memang tidak lebih dari kali elemen demi elemen, lalu jumlah.

Skor-skor ini belum bisa langsung dipakai untuk merata-ratakan, karena jumlahnya bukan 1. Langkah 2 menormalisasi skor menjadi *attention weight*, bobot yang semuanya positif dan berjumlah 1.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.9.png" width="700">

**Figure 3.9** Langkah 2: skor $\omega$ dinormalisasi menjadi bobot attention $\alpha$ yang berjumlah 1.

In [6]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


Cara paling sederhana adalah membagi setiap skor dengan jumlah semua skor, 6.5617. Skor pertama, misalnya, menjadi $0.9544 / 6.5617 = 0.1455$. Dengan cara ini perbandingan antarbobot sama persis dengan perbandingan antarskor: bobot "journey" 2.115 kali bobot "one" ($0.2278 / 0.1077$), sama dengan $1.4950 / 0.7070$.

Masalahnya, skor negatif akan menghasilkan bobot negatif, padahal rata-rata berbobot butuh bobot positif. Buku juga menyebut bahwa dalam praktik softmax lebih dianjurkan, karena lebih baik menangani nilai ekstrem dan memberi sifat gradien yang lebih baik saat training. Sebelum lanjut, perhatikan satu detail kecil di output: jumlahnya tercetak `tensor(1.0000)`, bukan `tensor(1.)`. Kita akan kembali ke sini sebentar lagi.

In [7]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


Softmax memberi `[0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]`. Setiap skor dipakai sebagai pangkat dari $e$ sebelum dibagi dengan jumlahnya, sehingga hasilnya selalu positif. Ada efek lain yang lebih halus: perbandingan dua bobot kini ditentukan oleh *selisih* skornya, bukan rasionya. Bobot "journey" dibanding bobot "one" adalah $e^{1.4950 - 0.7070} = e^{0.788} = 2.199$, dan memang $0.2379 / 0.1082 = 2.199$. Akibatnya, token dengan skor tertinggi mendapat porsi sedikit lebih besar daripada dengan pembagian biasa: 0.2379, bukan 0.2278.

In [8]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


`torch.softmax` memberi angka yang sama persis dengan `softmax_naive`. Lalu kenapa tidak memakai versi naif saja? Buku menyebut versi naif bisa bermasalah secara numerik, misalnya overflow untuk input yang besar, tapi tidak menunjukkannya. Sel tambahan berikut menunjukkannya, sekaligus menjawab teka-teki `1.0000` dan `1.` tadi.

In [9]:
x_big = torch.tensor([1000.0, 1001.0, 999.0])
print("softmax_naive(x_big)       :", softmax_naive(x_big))
print("torch.softmax(x_big)       :", torch.softmax(x_big, dim=0))
print("softmax_naive(x_big - max) :", softmax_naive(x_big - x_big.max()))
print("float32 exp overflow mulai :", torch.finfo(torch.float32).max, "-> ln =", round(torch.log(torch.tensor(torch.finfo(torch.float32).max)).item(), 2))
print("sum normalisasi biasa      :", attn_weights_2_tmp.sum().item())
print("sum torch.softmax          :", attn_weights_2.sum().item())


softmax_naive(x_big)       : tensor([nan, nan, nan])
torch.softmax(x_big)       : tensor([0.2447, 0.6652, 0.0900])
softmax_naive(x_big - max) : tensor([0.2447, 0.6652, 0.0900])
float32 exp overflow mulai : 3.4028234663852886e+38 -> ln = 88.72
sum normalisasi biasa      : 1.0000001192092896
sum torch.softmax          : 1.0


Dengan input sekitar 1,000, versi naif menghasilkan `nan` di ketiga posisi. Penyebabnya: float32 hanya bisa menyimpan angka sampai sekitar $3.4 \times 10^{38}$, sehingga `exp` sudah menjadi tak hingga untuk input di atas $\ln(3.4 \times 10^{38}) \approx 88.72$. $e^{1000}$ menjadi tak hingga, dan tak hingga dibagi tak hingga adalah `nan`.

`torch.softmax` tetap memberi `[0.2447, 0.6652, 0.0900]`, dan versi naif memberi angka yang persis sama setelah input dikurangi nilai maksimumnya. Itulah trik standar untuk menstabilkan softmax. Trik itu sah karena softmax tidak berubah kalau semua input digeser dengan konstanta yang sama, $\text{softmax}(x - c) = \text{softmax}(x)$: pembilang dan penyebut sama-sama dikali $e^{-c}$, lalu faktor itu saling meniadakan.

Dua baris terakhir menjawab teka-teki format tadi. Jumlah hasil pembagian biasa adalah 1.0000001192092896, tepat $1 + 2^{-23}$, angka float32 terdekat di atas 1. Karena tidak tepat 1, PyTorch mencetaknya `1.0000`. Jumlah hasil softmax tepat 1.0, sehingga dicetak `1.`. Bedanya hanya pembulatan di bit terakhir, bukan kesalahan.

Bobotnya sudah siap. Tinggal langkah 3: kalikan setiap vektor input dengan bobotnya, lalu jumlahkan semuanya.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.10.png" width="700">

**Figure 3.10** Langkah 3: context vector $z^{(2)}$ adalah jumlah semua vektor input, masing-masing dikalikan dengan bobot attention-nya.

In [10]:
query = inputs[1] # 2nd input token is the query

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


Inilah context vector pertama kita: $z^{(2)} = $ `[0.4419, 0.6515, 0.5683]`. Karena bobotnya positif dan berjumlah 1, setiap komponen $z^{(2)}$ adalah rata-rata berbobot dari kolom yang bersesuaian, sehingga nilainya pasti berada di antara nilai terkecil dan terbesar kolom itu. Kolom pertama input berkisar dari 0.05 sampai 0.77, dan 0.4419 ada di antaranya.

Arah tarikannya juga bisa ditebak. "journey" dan "starts" bersama-sama memegang hampir setengah bobot ($0.2379 + 0.2333 = 0.4712$), sehingga $z^{(2)}$ paling dekat dengan kedua embedding itu. Embedding "journey" yang tadinya berdiri sendiri kini membawa sedikit informasi dari seluruh kalimat.

### 3.3.2 Computing attention weights for all input tokens

Sejauh ini kita hanya menghitung untuk satu query. Sekarang ketiga langkah yang sama diulang untuk keenam token sekaligus. Hasil langkah 2 akan berupa matriks bobot $6 \times 6$: baris ke-$i$ berisi bobot untuk query ke-$i$, dan baris kedua adalah bobot "journey" yang baru saja kita hitung (figure 3.11 dan 3.12).

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.11.png" width="700">

**Figure 3.11** Matriks bobot attention untuk semua pasangan token. Baris yang disorot adalah bobot untuk query "journey". Angka di gambar ditampilkan dengan dua desimal.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.12.png" width="700">

**Figure 3.12** Tiga langkah yang sama -- skor, bobot, context vector -- kini untuk semua input sekaligus. Langkah 1 mendapat satu loop tambahan.

In [11]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


Matriks skor ini simetris: skor "journey" terhadap "starts" sama dengan skor "starts" terhadap "journey", karena dot product tidak peduli urutan. Diagonalnya adalah kuadrat panjang setiap embedding, dan baris kedua sama dengan `attn_scores_2`.

Ada satu hal yang langsung terbaca dari angka-angka ini: token *tidak* selalu memberi skor tertinggi kepada dirinya sendiri. Hanya "Your" dan "journey" yang begitu. Baris "starts", misalnya, memberi skor 1.4754 kepada "journey", lebih tinggi daripada skor untuk dirinya sendiri (1.4570). Baris "with" dan "step" juga paling tinggi ke "journey". Baris "one" memperlihatkan efek yang sama dari arah lain: skornya untuk "starts" (0.7154) lebih tinggi daripada untuk dirinya sendiri (0.6654).

Penyebabnya adalah sifat dot product yang tadi kita catat. Dot product adalah hasil kali tiga hal: panjang vektor pertama, panjang vektor kedua, dan cosine similarity keduanya. Jadi vektor yang panjang ikut mendapat skor besar, walaupun arahnya bukan yang paling mirip. "journey" adalah vektor terpanjang di kalimat ini (diagonalnya 1.4950, yang terbesar), disusul "starts" (1.4570), sedangkan "one" termasuk yang pendek (0.6654). Di contoh ini, selisih panjang seperti itu cukup untuk mengalahkan kemiripan arah.

Inilah salah satu alasan bobot yang bisa dilatih diperkenalkan di 3.4. Model perlu kebebasan untuk mempelajari sendiri kemiripan seperti apa yang berguna, bukan sekadar menerima kemiripan bawaan embedding.

In [12]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


`inputs @ inputs.T` memberi matriks yang sama persis dengan dua loop bersarang tadi. Perkalian matriks ini adalah bentuk ringkas dari $\Omega = XX^\top$: setiap elemen hasilnya adalah dot product satu baris $X$ dengan satu baris lain. Bedanya ada di kecepatan. Loop Python mengerjakan $T^2$ dot product satu per satu, sedangkan `@` menyerahkan seluruh pekerjaan ke satu operasi matriks yang sudah dioptimalkan. Untuk enam token selisihnya tidak terasa, tapi pada konteks GPT-2 yang panjangnya 1,024 token, loop itu berarti 1,048,576 putaran interpreter untuk satu kalimat.

In [13]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


`dim=-1` berarti softmax dikerjakan di sepanjang dimensi terakhir, yaitu per baris, sehingga setiap baris berjumlah 1. Baris kedua, `[0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]`, identik dengan bobot "journey" yang kita hitung di 3.3.1. Pola skor tadi juga terbawa: baris "starts" memberi bobot terbesar kepada "journey" (0.2369), sedikit di atas bobot untuk dirinya sendiri (0.2326).

In [14]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)

print("All row sums:", attn_weights.sum(dim=-1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


Baris pertama output menjumlahkan enam angka yang diketik ulang dari baris kedua, yang sudah dibulatkan ke empat desimal; hasilnya kebetulan tepat 1.0. Baris kedua menjumlahkan tensor bobot yang sebenarnya, dan keenam barisnya tercetak `1.0000`. Seperti di 3.3.1, format itu berarti setidaknya satu jumlah tidak tepat 1. Karena tetap dibulatkan menjadi `1.0000`, selisihnya pasti di bawah 0.00005, terlalu kecil untuk berarti apa-apa bagi perhitungan selanjutnya.

In [15]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


Langkah 3 untuk semua token juga cukup satu perkalian matriks, `attn_weights @ inputs`. Setiap baris hasilnya adalah context vector satu token, dan baris kedua, `[0.4419, 0.6515, 0.5683]`, adalah $z^{(2)}$ yang sudah kita kenal.

Bandingkan keenam baris ini dengan input aslinya. Context vector jauh lebih seragam: kolom pertama input tersebar dari 0.05 sampai 0.77, sedangkan kolom pertama context vector hanya dari 0.4177 sampai 0.4671, kira-kira 15 kali lebih sempit. Itu wajar, karena setiap baris adalah rata-rata dari input yang sama, dengan bobot yang tidak jauh dari rata. Tanpa bobot yang bisa dilatih, attention di sini lebih banyak meratakan daripada memilih. Bagian berikutnya memberi model kendali atas *apa* yang dirata-ratakan dan dengan bobot berapa.

In [16]:
print("Previous 2nd context vector:", context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


Hasil loop di 3.3.1 sama persis dengan baris kedua versi matriks, jadi kedua implementasi konsisten. Self-attention sederhana sudah lengkap. Sekarang kita tambahkan bagian yang membuatnya bisa belajar.

## 3.4 Implementing self-attention with trainable weights

Varian berikutnya adalah mekanisme yang benar-benar dipakai di transformer asli dan di GPT, yang disebut *scaled dot-product attention*. Tiga langkahnya tetap sama: skor, bobot, context vector. Yang baru adalah tiga matriks bobot yang bisa dilatih, $W_q$, $W_k$, dan $W_v$. Setiap embedding input dikalikan dengan ketiganya, sehingga setiap token punya tiga wujud: *query*, *key*, dan *value*.

Nama-nama itu dipinjam dari dunia pencarian informasi dan basis data. Query adalah apa yang sedang dicari oleh sebuah token. Key adalah semacam label pada setiap token, yang dicocokkan dengan query untuk menghasilkan skor. Value adalah isi yang benar-benar diambil kalau sebuah token dianggap relevan. Di 3.3, ketiga peran itu dimainkan oleh embedding yang sama, sehingga kemiripan sepenuhnya ditentukan oleh embedding bawaan. Dengan tiga proyeksi terpisah, skor dihitung antara query dan key, dan context vector menjadi rata-rata berbobot dari value. Ketiga matriks ini diperbarui selama training (Chapter 5), sehingga model belajar sendiri ukuran kemiripan yang berguna dan isi apa yang perlu diambil.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.13.png" width="700">

**Figure 3.13** Posisi kita di peta bab: self-attention sederhana sudah selesai, sekarang ditambahkan bobot yang bisa dilatih. Causal attention dan multi-head attention menyusul.

### 3.4.1 Computing the attention weights step by step

Seperti tadi, kita kerjakan dulu untuk satu query, "journey", langkah demi langkah, sebelum membungkus semuanya menjadi sebuah kelas.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.14.png" width="700">

**Figure 3.14** Langkah pertama dengan bobot yang bisa dilatih: input $x^{(i)}$ dikalikan dengan $W_q$, $W_k$, dan $W_v$ menjadi query, key, dan value. Token kedua tetap menjadi query.

In [17]:
x_2 = inputs[1] # second input element
d_in = inputs.shape[1] # the input embedding size, d=3
d_out = 2 # the output embedding size, d=2

`d_in = 3` adalah lebar embedding input, dan `d_out = 2` adalah lebar hasil proyeksi. Di GPT keduanya biasanya sama; untuk GPT-2 small, keduanya 768. Di sini sengaja dibuat berbeda supaya perubahan ukuran tensor mudah diikuti: setiap vektor masuk dengan 3 angka dan keluar dengan 2.

In [18]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

Ketiga matriks berukuran $3 \times 2$ dan diisi `torch.rand`, yaitu angka acak seragam antara 0 dan 1, dengan seed 123. `requires_grad=False` hanya supaya output lebih bersih; kalau matriks ini akan dilatih, flag itu harus `True`.

Satu hal soal istilah, karena kata "bobot" kini punya dua arti. *Weight parameter* adalah angka-angka di dalam jaringan yang dilatih, seperti $W_q$, $W_k$, dan $W_v$, dan nilainya tetap selama satu forward pass. *Attention weight* adalah bobot dinamis yang menentukan seberapa besar sumbangan setiap token ke context vector, dan dihitung ulang untuk setiap input.

In [19]:
query_2 = x_2 @ W_query # _2 because it's with respect to the 2nd input element
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


Query untuk "journey" adalah $q^{(2)} = x^{(2)} W_q = $ `[0.4306, 1.4551]`. Embedding 3 dimensi kini menjadi 2 dimensi, karena $W_q$ punya dua kolom.

In [20]:
keys = inputs @ W_key
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


Walaupun yang kita kejar hanya $z^{(2)}$, key dan value untuk *semua* token tetap dibutuhkan: skor query "journey" dihitung terhadap key setiap token, dan context vector-nya diambil dari value setiap token. Keenam token diproyeksikan dalam satu perkalian matriks, `(6, 3) @ (3, 2) -> (6, 2)`.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.15.png" width="700">

**Figure 3.15** Langkah skor: dot product antara query $q^{(2)}$ dan setiap key $k^{(j)}$, bukan lagi antara embedding mentah.

In [21]:
keys_2 = keys[1] # Python starts index at 0
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


Skor "journey" terhadap dirinya sendiri kini $\omega_{22} = q^{(2)} \cdot k^{(2)} = 1.8524$. Perhatikan bahwa yang dikalikan adalah query milik "journey" dengan key milik "journey", dua vektor yang berbeda, sehingga skor dengan diri sendiri tidak lagi otomatis sama dengan kuadrat panjang vektor. Sel berikut menghitung keenam skor sekaligus.

In [22]:
attn_scores_2 = query_2 @ keys.T # All attention scores for given query
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


Baris skor lengkap untuk query "journey" adalah `[1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440]`, dengan elemen kedua sama dengan $\omega_{22}$ tadi. Polanya masih mirip 3.3: "journey" dan "starts" tetap teratas, "one" tetap terbawah. Itu wajar, karena belum ada yang dipelajari: $W_q$ dan $W_k$ masih berisi angka acak dari `torch.rand`, semuanya positif, sehingga query dan key hanyalah campuran positif dari embedding yang sama, dan pola kemiripan lamanya sebagian besar ikut terbawa. Bedanya, skor sekarang ditentukan oleh proyeksi, dan training bisa mengubah proyeksi itu.

Sebelum skor diubah menjadi bobot, ada satu langkah baru: skor dibagi dulu dengan akar dimensi key.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.16.png" width="700">

**Figure 3.16** Skor attention dinormalisasi dengan softmax menjadi bobot attention. Sebelum softmax, skor dibagi dengan $\sqrt{d_k}$.

In [23]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


Bobot untuk query "journey" adalah `[0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820]`. Di sini $d_k = 2$, jadi skor dibagi $\sqrt{2} = 1.4142$ sebelum softmax. Repo menulis `keys.shape[1]`, sedangkan buku menulis `keys.shape[-1]`; untuk `keys` berukuran `(6, 2)` keduanya bernilai 2.

Buku menjelaskan alasan pembagian ini dalam satu paragraf, berupa rantai tiga klaim. Kalau dimensinya besar, dot product ikut membesar. Skor yang besar membuat softmax hampir seperti fungsi tangga, dengan hampir semua bobot jatuh ke satu token. Dan di daerah itu gradien softmax mendekati nol, sehingga training melambat. Dengan $d_k = 2$ efeknya belum terasa, jadi Under the hood 1 menguji ketiga mata rantai itu pada dimensi yang lebih realistis.

### Under the hood 1 -- kenapa skor dibagi $\sqrt{d_k}$

Titik berangkatnya adalah asumsi dari paper transformer: setiap komponen $q$ dan $k$ adalah angka acak yang saling bebas, dengan rata-rata 0 dan varians 1. Dot product $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ adalah jumlah $d_k$ suku seperti itu, sehingga variansnya $d_k$ dan simpangan bakunya $\sqrt{d_k}$. Kalau ini benar, membagi skor dengan $\sqrt{d_k}$ mengembalikan simpangan bakunya ke sekitar 1, berapa pun dimensinya.

Sel berikut menguji hal itu untuk enam ukuran $d_k$, dari 2 sampai 4,096. Untuk setiap ukuran, ia mengambil 1,000 query acak, masing-masing dengan 6 key acaknya sendiri (sama dengan jumlah token di contoh kita), lalu mengukur tiga hal, sekali tanpa pembagi dan sekali dengan pembagi:

- simpangan baku skor, untuk dibandingkan dengan $\sqrt{d_k}$;
- rata-rata bobot terbesar sesudah softmax, untuk melihat seberapa tajam distribusinya (kalau rata, nilainya $1/6 \approx 0.17$; kalau semua bobot jatuh ke satu token, 1);
- $\|J\|$, ukuran besar matriks $J = \text{diag}(p) - pp^\top$, yaitu akar dari jumlah kuadrat semua elemennya (disebut norma Frobenius). Matriks ini adalah turunan output softmax $p$ terhadap input softmax, yang kita sebut *logit*: tanpa pembagi logit-nya adalah skor mentah $s$, dengan pembagi logit-nya $s / \sqrt{d_k}$. $\|J\|$ mengukur seberapa besar output softmax masih bisa berubah kalau logit-nya digeser sedikit. Kalau mendekati nol, softmax sudah *tersaturasi*: ia nyaris tidak bereaksi, dan gradien yang lewat di situ ikut mengecil.

Sel ini memakai generator acaknya sendiri, sehingga urutan angka acak bab tidak terganggu.

In [24]:
uth_gen = torch.Generator().manual_seed(123)
T_tok, N_q = 6, 1000
uth1 = []
print(f"{'d_k':>5} {'std(q.k)':>9} {'sqrt(d_k)':>9} | tanpa skala: {'max w':>6} {'||J||':>7} | dibagi sqrt(d_k): {'max w':>6} {'||J||':>7}")
for dk in [2, 16, 64, 256, 768, 4096]:
    q_r = torch.randn(N_q, 1, dk, generator=uth_gen)
    k_r = torch.randn(N_q, T_tok, dk, generator=uth_gen)
    s_r = (q_r @ k_r.transpose(1, 2)).squeeze(1)                    # (N_q, T_tok) skor mentah
    row = [dk, s_r.std().item(), dk ** 0.5]
    for scale in (1.0, dk ** 0.5):
        p_r = torch.softmax(s_r / scale, dim=-1)
        J_r = torch.diag_embed(p_r) - p_r.unsqueeze(-1) * p_r.unsqueeze(-2)   # dp/dlogit, logit = s / scale
        row += [p_r.max(dim=-1).values.mean().item(), J_r.flatten(1).norm(dim=-1).mean().item()]
    uth1.append(row)
    print(f"{row[0]:>5} {row[1]:>9.3f} {row[2]:>9.3f} | {'':>12} {row[3]:>6.4f} {row[4]:>7.4f} |"
          f" {'':>17} {row[5]:>6.4f} {row[6]:>7.4f}")


  d_k  std(q.k) sqrt(d_k) | tanpa skala:  max w   ||J|| | dibagi sqrt(d_k):  max w   ||J||
    2     1.396     1.414 |              0.4580  0.3684 |                   0.3822  0.3800
   16     4.065     4.000 |              0.7715  0.2436 |                   0.4170  0.3844
   64     8.078     8.000 |              0.8885  0.1379 |                   0.4217  0.3849
  256    16.077    16.000 |              0.9449  0.0746 |                   0.4179  0.3865
  768    27.517    27.713 |              0.9653  0.0488 |                   0.4165  0.3857
 4096    64.894    64.000 |              0.9860  0.0197 |                   0.4250  0.3848


Tabel memperlihatkan ketiga mata rantai itu, dengan satu batas: yang kita ukur adalah seberapa besar softmax masih meneruskan gradien, bukan kecepatan training itu sendiri. Bahwa gradien yang mengecil memperlambat training adalah langkah terakhir argumen buku, dan langkah itu tidak diuji di sini.

Pertama, skor memang membesar seperti $\sqrt{d_k}$. Di $d_k = 64$ simpangan baku skor 8.078, hampir tepat $\sqrt{64} = 8$; di $d_k = 768$, ukuran embedding GPT-2 small, nilainya 27.517 dibanding $\sqrt{768} = 27.713$.

Kedua, tanpa pembagi, softmax berubah menjadi hampir fungsi tangga. Bobot terbesar rata-rata naik dari 0.4580 di $d_k = 2$ menjadi 0.9860 di $d_k = 4096$. Pada dimensi besar hampir seluruh bobot jatuh ke satu token, tidak peduli apakah token itu benar-benar paling relevan atau hanya kebetulan mendapat skor sedikit lebih tinggi.

Ketiga, softmax yang tersaturasi nyaris tidak meneruskan gradien. Tanpa pembagi, $\|J\|$ turun dari 0.3684 menjadi 0.0197, sekitar 19 kali lebih kecil. Karena di versi ini logit-nya adalah skor mentah, angka itu sekaligus mengukur turunan terhadap skor.

Dengan pembagi, gambarannya sama sekali lain. Bobot terbesar tetap di antara 0.38 dan 0.43, dan $\|J\|$ terhadap logit-nya bertahan di 0.3800 sampai 0.3865 untuk *semua* ukuran. **Pembagian dengan $\sqrt{d_k}$ menjaga softmax tetap bekerja di daerah yang sama, berapa pun dimensinya.**

Ada satu nuansa yang baru relevan di 3.6. Di `MultiHeadAttention`, yang dipakai sebagai $d_k$ adalah lebar satu head, bukan lebar embedding. Untuk GPT-2 small itu $768 / 12 = 64$, jadi pembaginya $\sqrt{64} = 8$. Baris $d_k = 64$ memperlihatkan bahwa tanpa pembagi softmax sudah cukup tajam di ukuran itu (bobot terbesar 0.8885, dibanding 0.4217 dengan pembagi). Hati-hati membandingkan kedua angka $\|J\|$ di baris itu, 0.1379 dan 0.3849, karena masing-masing diukur terhadap logit-nya sendiri. Terhadap skor mentah, turunan versi berskala adalah $0.3849 / \sqrt{64} = 0.0481$, malah lebih kecil daripada 0.1379 milik versi tanpa pembagi. Jadi yang dicegah oleh pembagian ini bukan "gradien kecil" secara umum, melainkan saturasi. Faktor $1/\sqrt{d_k}$ adalah pengali tetap yang sama untuk semua input, sedangkan saturasi bergantung pada input: begitu softmax hampir *one-hot*, artinya seluruh bobot praktis jatuh ke satu token, nyaris tidak ada sinyal yang bisa mengubah token mana yang dipilih.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.17.png" width="700">

**Figure 3.17** Langkah terakhir: value vector ditimbang dengan bobot attention, lalu dijumlahkan menjadi context vector $z^{(2)}$.

In [25]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


Context vector "journey" kini $z^{(2)} = $ `[0.3061, 0.8210]`. Rumusnya sama dengan di 3.3, bobot dikali vektor lalu dijumlahkan, tapi yang ditimbang sekarang adalah value, bukan embedding mentah. Karena itu lebar context vector mengikuti lebar value (`d_out = 2`), bukan lebar input.

### 3.4.2 Implementing a compact self-attention Python class

Langkah-langkah tadi sekarang dibungkus menjadi satu `nn.Module`, bentuk yang nanti bisa dipasang ke arsitektur LLM di Chapter 4. Repo menaruh Listing 3.1 dan contoh pemakaiannya di satu sel, sedangkan buku memisahkannya.

In [26]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


Enam context vector, masing-masing 2 dimensi, satu untuk setiap token. Baris kedua, `[0.3061, 0.8210]`, sama persis dengan `context_vec_2` yang kita hitung langkah demi langkah. Ini bukan kebetulan: seed 123 diulang, dan `__init__` mengambil tiga `torch.rand(3, 2)` dalam urutan yang sama seperti di 3.4.1 (query, key, value), sehingga bobotnya identik. Keterangan `grad_fn=<MmBackward0>` muncul karena bobot kini berupa `nn.Parameter` dengan `requires_grad=True`, sehingga PyTorch mencatat operasinya untuk menghitung gradien nanti.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.18.png" width="700">

**Figure 3.18** Self-attention dalam bentuk matriks: input $X$ dikalikan dengan $W_q$, $W_k$, dan $W_v$ menjadi $Q$, $K$, dan $V$; bobot attention dihitung dari $Q$ dan $K$, lalu dipakai untuk menimbang $V$ menjadi context vector $Z$.

In [27]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


`SelfAttention_v2` mengganti `nn.Parameter(torch.rand(...))` dengan `nn.Linear(..., bias=False)`. Tanpa bias, `nn.Linear` hanyalah perkalian matriks, jadi secara matematis tidak ada yang berubah. Menurut buku, keuntungannya ada di inisialisasi: `nn.Linear` membawa skema inisialisasi bobot yang membuat training lebih stabil.

Seed di sel ini 789, jadi bobotnya juga berbeda dari `v1`. Yang mencolok adalah output-nya: nilainya kecil, sekitar `-0.07` dan `0.07`, dan keenam barisnya hampir sama. Penyebabnya inisialisasi tadi. `nn.Linear` mengisi bobotnya dengan angka acak seragam antara $-1/\sqrt{d_{in}}$ dan $1/\sqrt{d_{in}}$, di sini $\pm 0.577$, sehingga kira-kira setengahnya negatif, sementara `torch.rand` di `v1` semuanya positif. Proyeksi seperti ini menghasilkan query dan key yang kecil, skor yang kecil, softmax yang hampir rata, dan akhirnya context vector yang mendekati rata-rata semua value. Matriks bobot di 3.5.1 nanti memperlihatkannya langsung.

Di buku, **Exercise 3.1** muncul di titik ini: menyalin bobot `SelfAttention_v2` ke `SelfAttention_v1` sehingga keduanya memberi output yang sama. Solusinya di repo mendefinisikan ulang kedua kelas dan membuat ulang `sa_v2` dengan seed 123, padahal 3.5.1 memakai `sa_v2` dengan seed 789 dari sel di atas. Karena itu solusinya kita jalankan di bagian Exercises di akhir notebook, supaya notebook yang dijalankan dari atas ke bawah tetap menghasilkan angka-angka buku.

## 3.5 Hiding future words with causal attention

GPT menulis teks dari kiri ke kanan, satu token setiap langkah, dan saat menebak token berikutnya ia hanya boleh memakai token yang sudah ada. Masalahnya, saat training seluruh kalimat diproses sekaligus. Dengan self-attention dari 3.4, context vector "starts" ikut dibentuk dari "with", padahal "with" adalah token yang harus ditebak sesudah "starts". Model yang boleh mengintip jawaban tidak perlu belajar menebak.

*Causal attention*, disebut juga *masked attention*, menutup celah itu. Untuk setiap token, semua bobot ke token sesudahnya -- semua elemen di atas diagonal matriks bobot -- dibuat nol, lalu bobot yang tersisa dinormalisasi ulang supaya setiap baris tetap berjumlah 1 (figure 3.19).

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.19.png" width="700">

**Figure 3.19** Causal mask: bobot di atas diagonal ditutup, sehingga "journey" hanya bisa melihat "Your" dan dirinya sendiri. Setiap baris lalu dinormalisasi ulang.

### 3.5.1 Applying a causal attention mask

Buku menempuh dua rute. Rute pertama terdiri dari tiga langkah (figure 3.20): hitung bobot dengan softmax seperti biasa, kalikan dengan mask berisi nol dan satu, lalu normalisasi ulang setiap baris. Supaya tidak perlu membuat bobot baru, matriks query dan key diambil dari `sa_v2` (seed 789) di 3.4.2.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.20.png" width="700">

**Figure 3.20** Rute pertama: softmax atas skor, nolkan elemen di atas diagonal, lalu normalisasi ulang setiap baris.

In [28]:
# Reuse the query and key weight matrices of the
# SelfAttention_v2 object from the previous section for convenience
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


Inilah matriks bobot $6 \times 6$ sebelum mask, dan di sini terlihat akibat inisialisasi `nn.Linear` tadi: semua bobot berada di antara 0.1477 dan 0.2041, dekat sekali dengan $1/6 = 0.1667$. Setiap token membagi perhatiannya hampir rata. Hanya kolom pertama, "Your", yang sedikit unggul di semua baris.

In [29]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


`torch.tril` membuat matriks segitiga bawah. Angka 1 menandai posisi yang boleh dilihat, yaitu token itu sendiri dan semua token sebelumnya, sehingga baris ke-$t$ berisi tepat $t$ angka 1.

In [30]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


Perkalian elemen demi elemen menolkan semua posisi masa depan, tapi baris-barisnya kini tidak lagi berjumlah 1. Baris pertama hanya menyisakan 0.1921, sedangkan baris terakhir masih utuh, karena token terakhir memang boleh melihat semuanya.

In [31]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


Setelah setiap baris dibagi jumlahnya, semua baris kembali berjumlah 1. Baris pertama menjadi `[1.0000, 0, ...]`: token pertama tidak punya apa pun untuk diperhatikan selain dirinya sendiri. Baris "journey" menjadi `[0.5517, 0.4483]`, kira-kira $0.2041 / (0.2041 + 0.1659)$ dan $0.1659 / 0.3700$. Dari angka yang tercetak, pembagian itu memberi 0.5516 dan 0.4484; selisih di desimal keempat muncul karena kode membagi bobot yang belum dibulatkan. Perbandingan antarbobot yang tersisa tidak berubah oleh normalisasi ulang; yang berubah hanya skalanya.

Rute kedua lebih ringkas (figure 3.21). Alih-alih menolkan bobot *sesudah* softmax, skor di posisi masa depan diisi $-\infty$ *sebelum* softmax. Karena $e^{-\infty} = 0$, posisi itu otomatis mendapat bobot nol, dan softmax langsung menghasilkan baris yang berjumlah 1.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.21.png" width="700">

**Figure 3.21** Rute kedua yang lebih efisien: isi skor di atas diagonal dengan $-\infty$, lalu terapkan softmax. Normalisasi ulang tidak diperlukan lagi.

In [32]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


Ini skor mentah dengan `-inf` di atas diagonal. Mask di sini dibuat dengan `torch.triu(..., diagonal=1)`, sehingga angka 1 menandai posisi yang *ditutup* -- kebalikan dari `mask_simple`, yang memakai 1 untuk posisi yang boleh dilihat. Konvensi `mask_simple` itu, 1 untuk posisi yang boleh dilihat, akan kita temui lagi di akhir bab, di dalam checkpoint GPT-2 di Hugging Face. `masked_fill` (tanpa garis bawah di ujung) mengembalikan tensor baru, jadi `attn_scores` sendiri tidak berubah.

In [33]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


Bobotnya sama dengan hasil rute pertama, termasuk `[0.5517, 0.4483]` untuk "journey".

Ada dua perbedaan kecil antara buku dan repo yang perlu diperiksa: softmax dengan `dim=1` (buku) lawan `dim=-1` (repo) di sel ini, dan `keys.shape[-1]` (buku) lawan `keys.shape[1]` (repo) di 3.4.1. Sel tambahan berikut memeriksa apakah perbedaan itu berpengaruh, dan apakah kedua rute masking benar-benar memberi hasil yang sama.

In [34]:
w_book = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)      # ejaan buku hlm. 78
print("softmax dim=1 (buku) == dim=-1 (repo) :", torch.equal(w_book, attn_weights))
print("keys.shape[1] == keys.shape[-1]       :", keys.shape[1] == keys.shape[-1], tuple(keys.shape))
print("rute 1 (renorm) vs rute 2 (-inf), max |selisih| :", (masked_simple_norm - attn_weights).abs().max().item())


softmax dim=1 (buku) == dim=-1 (repo) : True
keys.shape[1] == keys.shape[-1]       : True (6, 2)
rute 1 (renorm) vs rute 2 (-inf), max |selisih| : 2.9802322387695312e-08


Untuk tensor dua dimensi, `dim=1` dan `dim=-1` menunjuk sumbu yang sama, dan hasilnya identik sampai bit terakhir. `keys.shape[1]` dan `keys.shape[-1]` juga sama-sama 2. Jadi perbedaan penulisan buku dan repo tidak mengubah apa pun di sini. Meski begitu, `dim=-1` adalah kebiasaan yang lebih aman: di Listing 3.3 nanti tensor bobot menjadi tiga dimensi (batch, token, token), dan `dim=1` akan menormalisasi di sepanjang sumbu yang salah.

Kedua rute masking berselisih paling banyak 2.98e-08. Hasilnya sama sampai pembulatan float32, tapi tidak identik bit demi bit, karena jalur aritmetikanya berbeda: rute pertama membagi hasil softmax dengan jumlah baris, rute kedua langsung menghitung softmax dengan suku nol di posisi masa depan.

Untuk rute pertama, buku menambahkan argumen aljabar. Softmax awalnya memang sempat melibatkan skor token masa depan, tapi normalisasi ulang menghapus pengaruh itu, sehingga tidak ada informasi masa depan yang bocor. Under the hood 2 tidak mengulang argumen itu, tapi menguji akibatnya secara langsung. Ujinya memakai rute kedua, yang baru saja kita lihat memberi bobot yang sama dengan rute pertama.

### Under the hood 2 -- causal mask benar-benar menutup masa depan

Pernyataan bahwa token tidak bisa melihat masa depan bisa diuji dari dua arah. Fungsi `causal_context` di sel berikut adalah resep 3.5.1 dalam satu fungsi -- skor, $-\infty$ di atas diagonal, softmax berskala, lalu rata-rata value -- dengan bobot `sa_v2`.

Uji pertama bersifat kasar. Embedding token 4-6 ("with", "one", "step") diganggu dengan noise acak bersimpangan baku 10, jauh lebih besar daripada nilai embedding itu sendiri, lalu kita periksa apakah context vector token 1-3 ikut bergeser. Uji kedua lebih teliti. Dengan autograd, sel ini menghitung seluruh turunan $\partial z_t / \partial x_s$: seberapa peka output token $t$ terhadap perubahan kecil pada input token $s$. Untuk causal attention, semua turunan terhadap token masa depan ($s > t$) harus tepat nol. Versi tanpa mask dihitung sebagai pembanding.

In [35]:
def causal_context(x, causal=True):
    q_c, k_c, v_c = sa_v2.W_query(x), sa_v2.W_key(x), sa_v2.W_value(x)
    s_c = q_c @ k_c.T
    if causal:
        n = x.shape[0]
        s_c = s_c.masked_fill(torch.triu(torch.ones(n, n), diagonal=1).bool(), -torch.inf)
    return torch.softmax(s_c / k_c.shape[-1]**0.5, dim=-1) @ v_c

x_future = inputs.clone()
x_future[3:] += 10 * torch.randn(3, 3, generator=torch.Generator().manual_seed(0))
z_base, z_pert = causal_context(inputs), causal_context(x_future)
uth2_shift = (z_base[3:] - z_pert[3:]).abs().max().item()
print("token 1-3 bit-identik setelah token 4-6 diganggu :", torch.equal(z_base[:3], z_pert[:3]))
print("token 4-6, max |selisih|                         :", round(uth2_shift, 4))

dep_causal = torch.autograd.functional.jacobian(causal_context, inputs).abs().sum(dim=(1, 3))
dep_full = torch.autograd.functional.jacobian(lambda x: causal_context(x, causal=False), inputs).abs().sum(dim=(1, 3))
print("sum |dz_t/dx_s|, baris t = output, kolom s = input (causal):")
print(dep_causal)
print("segitiga atas (s > t), max   :", dep_causal.triu(1).max().item())
print("tanpa mask, entri nol        :", int((dep_full == 0).sum()), "dari 36")


token 1-3 bit-identik setelah token 4-6 diganggu : True
token 4-6, max |selisih|                         : 12.5887
sum |dz_t/dx_s|, baris t = output, kolom s = input (causal):
tensor([[1.3988, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7712, 0.6285, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5307, 0.4332, 0.4362, 0.0000, 0.0000, 0.0000],
        [0.3857, 0.3443, 0.3449, 0.3236, 0.0000, 0.0000],
        [0.3035, 0.2768, 0.2771, 0.2632, 0.2781, 0.0000],
        [0.2699, 0.2323, 0.2330, 0.2147, 0.2389, 0.2041]])
segitiga atas (s > t), max   : 0.0
tanpa mask, entri nol        : 0 dari 36


Kedua uji memberi jawaban yang tegas.

Gangguan sebesar itu menggeser context vector token 4-6 sampai 12.5887, tapi context vector token 1-3 tetap identik bit demi bit (`torch.equal` bernilai `True`). Bukan sekadar mirip: tidak ada satu bit pun yang berubah.

Matriks turunannya berbentuk segitiga bawah murni. Nilai terbesar di atas diagonal adalah 0.0, nol yang eksak, bukan angka yang sangat kecil. Penyebabnya, $e^{-\infty}$ benar-benar nol, sehingga bobot ke token masa depan nol, dan jalur gradien lewat key maupun value token itu ikut terputus. Tanpa mask, tidak ada satu pun dari 36 entri yang bernilai nol: setiap token bergantung pada semua token.

Dua pola lain ikut terbaca dari matriks itu. Baris pertama hanya punya satu entri, 1.3988, yang terbesar di seluruh matriks, karena token pertama sepenuhnya bergantung pada dirinya sendiri. Makin ke bawah, kepekaan terhadap setiap sumber makin kecil (baris terakhir hanya 0.2041 sampai 0.2699), karena bobotnya dibagi ke lebih banyak token. **Jaminan inilah yang membuat training di Chapter 5 sah: tebakan di posisi $t$ tidak pernah bisa mengintip jawaban di posisi $t+1$.**

### 3.5.2 Masking additional attention weights with dropout

Ada satu cara lain menutup bobot attention, yang hanya dipakai selama training: *dropout*. Dropout menolkan sebagian unit secara acak, supaya model tidak terlalu bergantung pada unit tertentu dan tidak mudah menghafal data training; saat model dipakai, dropout dimatikan. Di transformer, dropout biasanya dipasang di salah satu dari dua tempat: sesudah bobot attention dihitung, atau sesudah bobot itu dikalikan dengan value. Buku memakai tempat pertama (figure 3.22), dengan rate 0.5 supaya efeknya terlihat jelas. GPT di bab-bab berikutnya memakai 0.1.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.22.png" width="700">

**Figure 3.22** Mask dropout acak diterapkan pada bobot causal attention, sehingga sebagian bobot ikut menjadi nol selama training.

In [36]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # dropout rate of 50%
example = torch.ones(6, 6) # create a matrix of ones

print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


Sebelum menerapkannya pada bobot attention, buku mencobanya dulu pada matriks berisi angka 1, dan mask di atas sama dengan yang dicetak buku elemen demi elemen (Under the hood 3 nanti memeriksanya secara otomatis). Ada dua hal yang terlihat. Pertama, yang dinolkan adalah 15 dari 36 elemen (41.7%), bukan tepat setengah, karena setiap elemen diundi sendiri-sendiri dengan peluang 0.5. Kedua, elemen yang selamat tidak lagi bernilai 1, melainkan 2. Elemen itu dikali $1/(1 - 0.5) = 2$ supaya nilai harapan setiap elemen tetap 1 ($0.5 \cdot 2 = 1$), sehingga besaran output rata-rata tidak berubah antara training dan pemakaian.

In [37]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


Dengan seed yang sama, dropout yang sama kini diterapkan pada bobot causal dari 3.5.1. Lihat baris kedua, baris "journey": nol semua. Kedua bobotnya kebetulan terkena dropout, sehingga pada forward pass ini context vector "journey" adalah vektor nol, dan attention tidak menyumbang apa pun untuk token itu. Baris pertama sebaliknya bernilai 2.0, yaitu bobot tunggalnya, 1.0, dikali 2. Jadi selama training, baris bobot attention *tidak* lagi berjumlah 1. Seberapa sering hal seperti ini terjadi diukur di Under the hood 3.

Buku dan repo mencatat bahwa output dropout bisa berbeda antar sistem operasi, dengan merujuk ke isu PyTorch [#121595](https://github.com/pytorch/pytorch/issues/121595). Di run ini, mask dari CPU tetap sama dengan mask buku. Under the hood 3 menunjukkan dari mana mask itu berasal, dan memperlihatkan langsung bahwa perangkat yang berbeda bisa memberi mask yang berbeda dari kode yang sama.

### Under the hood 3 -- dropout: RNG, backend, train vs eval

Sel pertama adalah semacam sidik jari dropout. Ia mengulang dropout buku (seed 123, rate 0.5, `ones(6, 6)`) lalu membandingkan hasilnya dengan dua acuan. Acuan pertama adalah mask yang dicetak buku di halaman 79, yang diketik ulang di sel sebagai `BOOK_MASK`. Acuan kedua dibuat dari undian biasa: dengan seed yang sama, ambil 36 angka acak seragam $u$ dalam presisi float64 lewat `torch.rand`, lalu pertahankan elemen yang $u < 0.5$. Kalau acuan kedua cocok, berarti mask dropout tidak lain adalah undian seragam dari generator acak CPU, satu angka untuk setiap elemen.

In [38]:
import platform
import torch

BOOK_MASK = torch.tensor([[2., 2., 0., 2., 2., 0.],
                          [0., 0., 0., 2., 0., 2.],
                          [2., 2., 2., 2., 0., 2.],
                          [0., 2., 2., 0., 0., 2.],
                          [0., 2., 0., 2., 0., 2.],
                          [0., 2., 2., 2., 2., 0.]])


def dropout_fingerprint():
    torch.manual_seed(123)
    out = torch.nn.Dropout(0.5)(torch.ones(6, 6))
    torch.manual_seed(123)
    u = torch.rand(6, 6, dtype=torch.float64)
    print(f"platform               : {platform.system()} {platform.machine()} | torch {torch.__version__}"
          f" | MKL {torch.backends.mkl.is_available()}")
    print(f"mask == book (p. 79)   : {torch.equal(out, BOOK_MASK)}")
    print(f"mask == 2*(u64 < 0.5)  : {torch.equal(out, (u < 0.5).float() * 2)}")
    print(f"zeros                  : {int((out == 0).sum())}/36 | row 0: {out[0].tolist()}")


dropout_fingerprint()

platform               : Linux x86_64 | torch 2.11.0+cu130 | MKL True
mask == book (p. 79)   : True
mask == 2*(u64 < 0.5)  : True
zeros                  : 15/36 | row 0: [2.0, 2.0, 0.0, 2.0, 2.0, 0.0]


Di konfigurasi run ini (Linux x86_64, torch 2.11.0+cu130), mask dropout identik dengan mask buku dan identik dengan `2 * (u64 < 0.5)`. Jadi `Dropout` di sini mengambil satu angka acak seragam float64 untuk setiap elemen dari generator CPU, mempertahankan elemen yang angkanya di bawah 0.5, dan mengalikan elemen yang dipertahankan dengan 2.

Kesimpulan ini hanya berlaku untuk konfigurasi yang diuji. Build atau perangkat lain bisa memakai jalur pembangkit acak yang berbeda untuk dropout, dan kalau itu terjadi, mask-nya berbeda walaupun kodenya sama persis. Perbedaan seperti itu adalah perbedaan RNG (*random number generator*) atau backend, bukan bug pada kode. Sel berikutnya memperlihatkan satu contohnya.

Sel kedua memeriksa perilaku dropout yang memengaruhi training. Ia mengulang dropout pada bobot causal sebanyak 10,000 kali, lalu mencatat jumlah setiap baris dan seberapa sering sebuah baris menjadi nol seluruhnya. Ia juga mengukur rate dropout yang sebenarnya pada tensor besar, memeriksa apakah `torch.no_grad()` mematikan dropout, melihat apa yang dilakukan `eval()`, dan terakhir membandingkan mask dari GPU (`device` yang dipilih di Setup) dengan mask buku.

In [39]:
torch.manual_seed(123)
drop = torch.nn.Dropout(0.5)
w_causal = attn_weights.detach()
trials = torch.stack([drop(w_causal) for _ in range(10_000)])
rs = trials.sum(dim=-1)
print("rowsum, undian pertama       :", rs[0])
print("rowsum, rata-rata 10k undian :", rs.mean(dim=0))
print("P(baris nol semua), terukur  :", (rs == 0).float().mean(dim=0))
print("P(baris nol semua), 0.5^t    :", 0.5 ** torch.arange(1, 7))
big = drop(torch.ones(1000, 1000))
print(f"ones(1000, 1000): frac nol {(big == 0).float().mean().item():.4f} | mean {big.mean().item():.4f}")
with torch.no_grad():
    print("no_grad + mode train, masih ada nol?   :", bool((drop(torch.ones(6, 6)) == 0).any()))
drop.eval()
print("mode eval, output == input (identitas)?:", torch.equal(drop(w_causal), w_causal))
drop.train()

if device.type == "cpu":
    print("akselerator: device = cpu, perbandingan backend dilewati")
else:
    torch.manual_seed(123)
    m_acc = torch.nn.Dropout(0.5)(torch.ones(6, 6, device=device)).cpu()
    print(f"akselerator {device.type}: mask == mask buku? {torch.equal(m_acc, BOOK_MASK)} | nol {int((m_acc == 0).sum())}/36")


rowsum, undian pertama       : tensor([2.0000, 0.0000, 2.0000, 0.9846, 0.7741, 1.3073])
rowsum, rata-rata 10k undian : tensor([1.0050, 0.9972, 0.9950, 1.0003, 1.0017, 1.0027])
P(baris nol semua), terukur  : tensor([0.4975, 0.2546, 0.1258, 0.0625, 0.0313, 0.0147])
P(baris nol semua), 0.5^t    : tensor([0.5000, 0.2500, 0.1250, 0.0625, 0.0312, 0.0156])
ones(1000, 1000): frac nol 0.5001 | mean 0.9998
no_grad + mode train, masih ada nol?   : True
mode eval, output == input (identitas)?: True
akselerator cuda: mask == mask buku? False | nol 14/36


Mulai dari baris pertama output. Undian pertama adalah dropout yang sama persis dengan sel buku di 3.5.2, karena seed-nya sama, jadi jumlahnya bisa dicek dari tensor di sana; baris keempat, misalnya, $0.4921 + 0.4925 = 0.9846$. Keenam jumlahnya 2.0000, 0.0000, 2.0000, 0.9846, 0.7741, dan 1.3073: ada baris yang menjadi 0 dan ada yang melampaui 1. **Saat training, jumlah satu baris bobot attention bukan 1; yang tetap 1 hanyalah rata-ratanya.** Dalam 10,000 undian, rata-rata setiap baris berada di antara 0.9950 dan 1.0050. Dropout menjaga nilai harapan, bukan jumlah per sampel.

Seberapa sering satu baris hilang seluruhnya? Baris ke-$t$ punya tepat $t$ bobot yang tidak nol, jadi peluang semuanya terkena dropout adalah $0.5^t$. Hasil ukurnya cocok: 0.4975 untuk baris pertama, 0.2546 untuk baris kedua, terus mengecil sampai 0.0147 untuk baris keenam, dibanding 0.5, 0.25, dan 0.0156 dari rumus. Artinya, context vector token pertama menjadi nol di sekitar separuh forward pass. Dengan rate 0.1 yang dipakai GPT nanti, peluang yang sama hanya 0.1.

Rate yang sebenarnya memang 0.5. Pada matriks $1000 \times 1000$, fraksi elemen yang nol 0.5001 dan rata-ratanya 0.9998; angka 15 dari 36 di sel buku hanyalah variasi sampel kecil.

Dua baris berikutnya menyimpan pelajaran yang paling praktis. Di dalam `torch.no_grad()`, dengan modul masih dalam mode train, dropout *tetap* menolkan elemen. `no_grad()` hanya menghentikan pencatatan gradien; yang mematikan dropout adalah `.eval()`, dan dalam mode itu output identik dengan input. Inilah sebabnya fungsi evaluasi di Chapter 5 memanggil `model.eval()` sebelum menghitung loss validasi. Kalau lupa, loss validasi ikut teracak dropout.

Baris terakhir membandingkan mask dari GPU. Dengan seed yang sama, CUDA memberi mask yang berbeda dari mask buku, dengan 14 dari 36 elemen nol. `torch.manual_seed` memang mengatur seed untuk semua device, tapi setiap backend punya generator acak dan kernel dropout sendiri. Kode yang sama menghasilkan mask yang sama sahnya, hanya berbeda. (Baris ini bergantung pada device: di mesin tanpa GPU, perbandingannya dilewati.)

Pelajaran praktisnya: kalau mask dropout di mesin lain berbeda dari buku, itu belum tanda ada yang salah dengan kode. Lain halnya kalau sebuah output tercetak tidak mungkin dihasilkan oleh kodenya sendiri, berapa pun seed-nya. Kasus seperti itu akan kita temui di Exercise 3.2, dan di sana aritmetika sederhana cukup untuk menunjukkan kemustahilannya.

### 3.5.3 Implementing a compact causal attention class

Semua bagian causal attention sudah ada; tinggal menyatukannya dalam satu kelas. Kelas ini harus bisa menerima *batch*, seperti yang dihasilkan data loader di Chapter 2: tensor berbentuk `(batch, num_tokens, d_in)`. Untuk simulasi, kalimat contoh kita diduplikasi menjadi batch berisi dua teks.

In [40]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # 2 inputs with 6 tokens each, and each token has embedding dimension 3

torch.Size([2, 6, 3])


Bentuknya `(2, 6, 3)`: dua teks, masing-masing enam token, dengan embedding 3 dimensi.

In [41]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # New
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape # New batch dimension b
        # For inputs where `num_tokens` exceeds `context_length`, this will result in errors
        # in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs
        # do not exceed `context_length` before reaching this forward method.
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


Listing 3.3 menggabungkan semua yang sudah kita bangun: proyeksi `nn.Linear` dari `SelfAttention_v2`, mask $-\infty$ dari 3.5.1, dan dropout dari 3.5.2. (Anotasi listing ini di buku menyebutnya pengembangan dari `SelfAttention_v1`, tapi karena proyeksinya `nn.Linear`, kelas ini sebenarnya meneruskan `SelfAttention_v2`.) Selain itu ada empat perubahan kecil dibanding kode di 3.5.1. Hanya yang pertama muncul karena dimensi batch; tiga lainnya menyangkut cara mask disimpan dan dipakai:

1. `keys.transpose(1, 2)` menggantikan `.T`, supaya hanya dua dimensi terakhir yang ditukar dan dimensi batch tetap di depan.
2. Mask disimpan dengan `register_buffer`. *Buffer* ikut pindah device bersama model, tapi tidak dihitung sebagai parameter dan tidak dilatih.
3. `masked_fill_`, dengan garis bawah di ujung, mengubah skor di tempatnya tanpa membuat tensor baru.
4. Mask dipotong menjadi `[:num_tokens, :num_tokens]`, sehingga teks yang lebih pendek dari `context_length` tetap bisa diproses.

Buku hanya mencetak bentuk output, `(2, 6, 2)`; repo juga mencetak isinya. Kedua teks dalam batch memberi context vector yang sama karena input-nya memang duplikat. Baris pertamanya, `[-0.4519, 0.2216]`, akan muncul lagi di 3.6.1. Keterangan `grad_fn` di ujung output kini `UnsafeViewBackward0`, bukan `MmBackward0` seperti di 3.4.2, karena perkalian matriks tiga dimensi dikerjakan PyTorch sebagai perkalian berbatch, lalu hasilnya dibentuk ulang.

Komentar di listing repo menyebut bahwa input yang lebih panjang dari `context_length` akan menimbulkan error di pembuatan mask. Sel tambahan berikut mencobanya dengan memberi 7 token kepada modul yang dibuat untuk 6. Error di sel ini memang disengaja.

In [42]:
ca(torch.ones(1, 7, 3))

RuntimeError: The size of tensor a (7) must match the size of tensor b (6) at non-singleton dimension 2

Error memang muncul, tapi tempatnya sedikit berbeda dari yang disebut komentar. Memotong buffer $6 \times 6$ dengan `self.mask[:7, :7]` tidak gagal; PyTorch diam-diam mengembalikan seluruh matriks $6 \times 6$. Yang gagal adalah langkah sesudahnya: `masked_fill_` tidak bisa memasangkan mask $6 \times 6$ itu dengan skor $7 \times 7$, dan berhenti dengan `RuntimeError: The size of tensor a (7) must match the size of tensor b (6)`.

Pelajarannya: karena buffer mask berukuran tetap, `context_length` adalah batas keras panjang input, bukan sekadar saran. Model GPT di Chapter 4 sampai 7 menjaga batas ini dengan memotong input sepanjang `context_length` sebelum masuk ke forward pass.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.23.png" width="700">

**Figure 3.23** Posisi kita di peta bab: self-attention sederhana, self-attention dengan bobot yang bisa dilatih, dan causal attention sudah selesai. Tinggal multi-head attention.

## 3.6 Extending single-head attention to multi-head attention

### 3.6.1 Stacking multiple single-head attention layers

Satu causal attention menghasilkan satu cara menimbang token: satu set query, key, dan value, sehingga satu pola bobot. Padahal dalam sebuah kalimat ada banyak jenis hubungan yang mungkin berguna sekaligus. *Multi-head attention* menjalankan beberapa causal attention berdampingan, masing-masing dengan $W_q$, $W_k$, dan $W_v$ sendiri, sehingga setiap *head* bisa mempelajari pola bobotnya sendiri. Output semua head lalu disambung menjadi satu (figure 3.24).

Cara paling langsung untuk membangunnya adalah membuat beberapa `CausalAttention` di dalam `nn.ModuleList`, menjalankan semuanya, lalu menyambung output-nya di dimensi terakhir.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.24.png" width="700">

**Figure 3.24** Multi-head attention dengan dua head: dua set matriks bobot menghasilkan dua matriks context vector, $Z_1$ dan $Z_2$, yang digabung menjadi $Z$.

In [43]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


Output-nya berbentuk `(2, 6, 4)`: dua head, masing-masing dengan `d_out = 2`, menghasilkan $2 \times 2 = 4$ kolom. Dua kolom pertama identik dengan output `CausalAttention` di 3.5.3 -- baris pertamanya `[-0.4519, 0.2216]` lagi -- dan ini bisa ditebak sebelum sel dijalankan. Seed 123 diulang, dan head pertama dibuat lebih dulu, sehingga ia mengambil bobot yang persis sama dengan `ca`. Kolom ketiga dan keempat berasal dari head kedua, yang bobotnya diambil dari angka acak berikutnya.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.25.png" width="700">

**Figure 3.25** `MultiHeadAttentionWrapper` dengan `num_heads=2`. Setiap head memakai `d_out=2`, dan kedua matriks context vector disambung di dimensi kolom menjadi lebar 4.

Di buku, **Exercise 3.2** muncul di sini: dengan `num_heads=2` tetap, ubah satu argumen supaya output-nya 2 dimensi, bukan 4. Solusinya kita jalankan di bagian Exercises, dan ternyata ada kejutan di output yang dicetak repo untuk solusi itu.

### 3.6.2 Implementing multi-head attention with weight splits

Wrapper tadi bekerja, tapi setiap head menghitung proyeksi dan attention-nya sendiri, satu per satu, lewat `[head(x) for head in self.heads]`. `MultiHeadAttention` menggantinya dengan satu proyeksi besar untuk query, satu untuk key, dan satu untuk value. Hasil proyeksi itu lalu dipotong menjadi beberapa head dengan dua operasi bentuk. `.view(b, num_tokens, num_heads, head_dim)` memecah dimensi terakhir menjadi jumlah head dan lebar setiap head, lalu `.transpose(1, 2)` memindahkan dimensi head ke depan, sehingga setiap head diperlakukan seperti anggota batch tersendiri. Attention semua head kemudian dihitung dengan satu perkalian matriks berbatch, output semua head digabung kembali, dan hasilnya dilewatkan ke satu lapisan linear tambahan, `out_proj` (figure 3.26). Seperti sebelumnya, repo menaruh Listing 3.5 dan pemakaiannya di satu sel.

In [44]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`,
        # this will result in errors in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


Output-nya berbentuk `(2, 6, 2)`. Perhatikan arti `d_out` yang berubah: di kelas ini `d_out` adalah lebar *total* semua head, sehingga `d_out=2` dengan 2 head berarti setiap head hanya selebar 1 (`head_dim = 1`). Angkanya juga berbeda dari wrapper, karena bobot diambil dalam bentuk dan urutan yang berbeda, dan ada `out_proj` tambahan. Lapisan itu pula yang menjelaskan keterangan `grad_fn=<ViewBackward0>` di ujung output: `out_proj` adalah langkah terakhir forward pass, dan lapisan linear yang menerima tensor tiga dimensi mengerjakannya sebagai matriks dua dimensi, lalu mengembalikan bentuknya dengan `view`. Apakah kedua kelas benar-benar menghitung hal yang sama akan kita uji langsung di Under the hood 4.

<img src="https://raw.githubusercontent.com/farrelrassya/Build-a-Large-Language-Model-From-Scratch/main/Folder-photos/03.Chapter03/figure3.26.png" width="700">

**Figure 3.26** Atas: wrapper dengan dua matriks query terpisah, $W_{q1}$ dan $W_{q2}$. Bawah: `MultiHeadAttention` dengan satu $W_q$ besar, yang hasilnya ($Q$) lalu dipecah menjadi $Q_1$ dan $Q_2$. Key dan value diperlakukan dengan cara yang sama.

Perkalian matriks empat dimensi di dalam `MultiHeadAttention` mudah terasa seperti sulap. Buku membongkarnya dengan tensor kecil berbentuk `(b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)`: satu teks, dua head, tiga token, dan lebar head 4.

In [45]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


`a @ a.transpose(2, 3)` hanya mengalikan dua dimensi terakhir, `(3, 4) @ (4, 3)`, dan mengulanginya untuk setiap pasangan batch dan head. Hasilnya berbentuk `(1, 2, 3, 3)`: satu matriks $3 \times 3$ untuk setiap head. Kedua matriks itu simetris, karena untuk setiap head yang dihitung adalah $AA^\top$, persis seperti $XX^\top$ di 3.3.2.

In [46]:
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("First head:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


Mengalikan setiap head secara terpisah memberi angka yang sama persis. Itulah yang terjadi di dalam `MultiHeadAttention`: satu operator `@` menggantikan loop atas semua head.

Buku menyebut `MultiHeadAttention` lebih efisien daripada wrapper, karena proyeksi query, key, dan value cukup dihitung sekali untuk semua head, bukan sekali per head. Buku juga menyebut kedua kelas mengimplementasikan konsep yang sama. Under the hood 4 menguji keduanya.

### Under the hood 4 -- wrapper vs `MultiHeadAttention`: sama secara matematis, beda biaya

Kesetaraan diuji dengan *transplantasi bobot*. Supaya ukurannya cocok, `MultiHeadAttention` di uji ini dibuat dengan `d_out=4`: dua head selebar 2, sama dengan dua head wrapper yang masing-masing `d_out=2`, sehingga keduanya menghasilkan output selebar 4. Bobot query, key, dan value dari kedua head wrapper lalu disalin ke matriks tunggal `MultiHeadAttention` secara berurutan: baris-baris pertama untuk head 1, berikutnya untuk head 2. `out_proj` diubah menjadi matriks identitas, sehingga tidak mengubah apa pun. Kalau kedua kelas memang fungsi yang sama, output-nya harus sama sampai pembulatan float.

Sel yang sama lalu membangun kedua kelas dalam ukuran GPT-2 small (embedding 768, 12 head, konteks 1,024 token), menghitung parameter dan buffer masing-masing, dan membandingkan `MultiHeadAttention` dengan `F.scaled_dot_product_attention` (SDPA), fungsi attention bawaan PyTorch yang mengerjakan beberapa langkah attention dalam satu *kernel*, yaitu satu rutin komputasi tingkat rendah yang menjalankan langkah-langkah itu sekaligus.

In [47]:
import time
import torch.nn.functional as F

torch.manual_seed(123)
wrap_t = MultiHeadAttentionWrapper(3, 2, 6, 0.0, num_heads=2)
mha_t = MultiHeadAttention(3, 4, 6, 0.0, num_heads=2)
with torch.no_grad():
    for name in ["W_query", "W_key", "W_value"]:
        getattr(mha_t, name).weight.copy_(torch.cat([getattr(h, name).weight for h in wrap_t.heads], dim=0))
    mha_t.out_proj.weight.copy_(torch.eye(4))
    mha_t.out_proj.bias.zero_()
    uth4_diff = (wrap_t(batch) - mha_t(batch)).abs().max().item()
print(f"transplantasi bobot, max |wrapper - MHA| : {uth4_diff:.2e}")

torch.manual_seed(123)
d_g, H_g, T_g = 768, 12, 1024
wrap_g = MultiHeadAttentionWrapper(d_g, d_g // H_g, T_g, 0.0, num_heads=H_g).eval()
mha_g = MultiHeadAttention(d_g, d_g, T_g, 0.0, num_heads=H_g).eval()
x_g = torch.randn(2, T_g, d_g)

def mha_sdpa(x):
    b, n, _ = x.shape
    split = lambda t: t.view(b, n, H_g, d_g // H_g).transpose(1, 2)
    q, k, v = split(mha_g.W_query(x)), split(mha_g.W_key(x)), split(mha_g.W_value(x))
    o = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    return mha_g.out_proj(o.transpose(1, 2).contiguous().view(b, n, d_g))

with torch.no_grad():
    uth4_sdpa = (mha_sdpa(x_g) - mha_g(x_g)).abs().max().item()
n_par = lambda m: sum(p.numel() for p in m.parameters())
n_buf = lambda m: sum(b.numel() for b in m.buffers())
print(f"parameter : wrapper {n_par(wrap_g):,} | MHA {n_par(mha_g):,} (out_proj {n_par(mha_g.out_proj):,})")
print(f"buffer    : wrapper {n_buf(wrap_g):,} | MHA {n_buf(mha_g):,} elemen mask")
print(f"proyeksi  : wrapper {3 * H_g} Linear kecil (768 -> 64) | MHA 3 Linear besar (768 -> 768) + out_proj")
print(f"SDPA (is_causal=True) vs MHA, max |selisih| : {uth4_sdpa:.2e}")


transplantasi bobot, max |wrapper - MHA| : 5.96e-08
parameter : wrapper 1,769,472 | MHA 2,360,064 (out_proj 590,592)
buffer    : wrapper 12,582,912 | MHA 1,048,576 elemen mask
proyeksi  : wrapper 36 Linear kecil (768 -> 64) | MHA 3 Linear besar (768 -> 768) + out_proj
SDPA (is_causal=True) vs MHA, max |selisih| : 1.94e-07


Setelah transplantasi, wrapper dan `MultiHeadAttention` berselisih paling banyak 5.96e-08, sebesar pembulatan float32. Selisihnya tidak tepat nol, kemungkinan karena perkalian matriks dengan bentuk lain melewati jalur komputasi lain di PyTorch, sehingga pembulatannya sedikit bergeser. **Kedua kelas memang menghitung fungsi yang sama;** yang berbeda hanya cara memotong pekerjaannya.

Perbedaannya baru terlihat di ukuran GPT-2 small. Wrapper punya 1,769,472 parameter, yaitu $3 \times 768^2$ untuk proyeksi query, key, dan value. `MultiHeadAttention` punya 2,360,064, dan selisihnya, 590,592, persis ukuran `out_proj`: $768^2$ bobot ditambah 768 bias. Buffer-nya berbeda lebih jauh. Wrapper menyimpan 12,582,912 elemen mask, yaitu 12 salinan identik mask $1024 \times 1024$, satu di setiap head (sekitar 50 MB dalam float32), sedangkan `MultiHeadAttention` cukup satu salinan, 1,048,576 elemen.

SDPA dengan bobot yang sama berselisih paling banyak 1.94e-07 dari `MultiHeadAttention`, jadi fungsi bawaan PyTorch menghitung hal yang sama dengan Listing 3.5.

Baris `proyeksi` di output memperlihatkan dari mana gagasan efisiensi buku berasal. Wrapper menjalankan 36 lapisan Linear kecil (768 ke 64), tiga untuk setiap head, lalu menghitung attention 12 kali lewat loop Python. `MultiHeadAttention` cukup memanggil 3 Linear besar (768 ke 768), ditambah `out_proj`, dan menghitung attention semua head dalam satu perkalian berbatch. Jumlah perkalian untuk query, key, dan value sama, $36 \times 768 \times 64 = 3 \times 768^2$; yang berubah adalah bentuknya: lebih sedikit pemanggilan, masing-masing lebih besar.

Apakah bentuk kerja seperti itu benar-benar lebih cepat? Sel berikut mengukur waktu ketiga versi di CPU. Waktu eksekusi berubah-ubah dari run ke run, jadi angkanya hanya berlaku untuk run ini.

In [48]:
def median_ms(fn, reps=15):
    ts = []
    with torch.no_grad():
        fn(x_g)
        for _ in range(reps):
            t0 = time.perf_counter()
            fn(x_g)
            ts.append((time.perf_counter() - t0) * 1e3)
    return sorted(ts)[reps // 2]

uth4 = {"wrap": median_ms(wrap_g), "mha": median_ms(mha_g), "sdpa": median_ms(mha_sdpa)}
print(f"latency CPU, median 15 run, batch 2 x {T_g} token, {torch.get_num_threads()} thread:")
print(f"  wrapper (12 x CausalAttention) : {uth4['wrap']:7.1f} ms")
print(f"  MultiHeadAttention             : {uth4['mha']:7.1f} ms  ({uth4['wrap'] / uth4['mha']:.2f}x lebih cepat)")
print(f"  MHA + SDPA fused               : {uth4['sdpa']:7.1f} ms  ({uth4['wrap'] / uth4['sdpa']:.2f}x lebih cepat)")


latency CPU, median 15 run, batch 2 x 1024 token, 1 thread:
  wrapper (12 x CausalAttention) :   227.6 ms
  MultiHeadAttention             :   380.2 ms  (0.60x lebih cepat)
  MHA + SDPA fused               :   134.9 ms  (1.69x lebih cepat)


Hasilnya berlawanan dengan dugaan. Wrapper butuh 227.6 ms, sedangkan `MultiHeadAttention` 380.2 ms. Label "0.60x lebih cepat" di output berarti `MultiHeadAttention` justru *lebih lambat*: wrapper sekitar 1.67 kali lebih cepat. Hanya SDPA yang menang, dengan 134.9 ms, sekitar 1.69 kali lebih cepat daripada wrapper dan 2.8 kali lebih cepat daripada `MultiHeadAttention`.

Sebagian selisih itu bisa dihitung dengan tangan. Kedua kelas mengerjakan proyeksi query, key, dan value yang sama besarnya, ditambah perkalian attention yang sama besarnya: sekitar 13.69 GFLOP (miliar operasi floating point) untuk batch ini, karena matriks skor $1024 \times 1024$ dihitung penuh dan mask tidak melewatkan pekerjaan apa pun. `MultiHeadAttention` mengerjakan *lebih banyak*. `out_proj` menambah satu perkalian $768 \times 768$ untuk setiap token, sekitar 2.42 GFLOP atau 17.6% lebih banyak. Tapi tambahan itu jauh dari cukup untuk menjelaskan waktu yang 67% lebih lama.

Sisanya tidak diukur di sini, jadi yang bisa kita tawarkan hanya dugaan. Pertama, PyTorch di run ini memakai satu thread CPU (`threads=1` di Setup), sehingga keuntungan 3 perkalian besar dibanding 36 perkalian kecil, yang biasanya datang dari membagi pekerjaan besar ke banyak core, tidak ikut bermain. Kedua, `MultiHeadAttention` menyimpan matriks skor semua head sekaligus, $2 \times 12 \times 1024 \times 1024$ angka atau sekitar 100 MB float32, sedangkan wrapper hanya memegang potongan satu head setiap kali. Yang pasti hanya ini: di CPU satu thread pada run ini, klaim efisiensi buku tidak terlihat. Runtime ini punya GPU, tapi sel pengukuran ini hanya berjalan di CPU, jadi kita tidak tahu apakah urutannya berbeda di GPU.

SDPA unggul di kedua perbandingan. Fungsi ini dirancang menggabungkan mask, softmax, dan perkalian dengan value di dalam satu kernel, sehingga matriks skor sebesar 100 MB itu tidak perlu ditulis utuh ke memori. Di run ini, memakai fungsi bawaan memberi percepatan yang jauh lebih besar daripada mengganti cara memotong head.

### Ukuran GPT-2 yang disebut buku

Buku menutup 3.6.2 dengan menyebut ukuran model GPT-2 asli. Menurut buku, GPT-2 terkecil punya **117 juta** parameter, 12 head attention, dan embedding 768 dimensi, sedangkan GPT-2 terbesar punya 1.5 miliar parameter, 25 head, dan embedding 1,600 dimensi. Angka-angka ini mudah diperiksa, dan kita memeriksanya dengan dua cara.

Cara pertama murni aritmetika dari arsitektur, tanpa koneksi internet. Bagian attention dihitung dengan `MultiHeadAttention` dari bab ini, sedangkan MLP dan LayerNorm mengikuti arsitektur yang dibangun di Chapter 4.

In [49]:
def gpt2_params(d, n_layers, n_heads, vocab=50257, ctx=1024):
    attn = MultiHeadAttention(d, d, ctx, 0.0, num_heads=n_heads, qkv_bias=True)
    p_attn = sum(p.numel() for p in attn.parameters())
    p_mlp = (d * 4 * d + 4 * d) + (4 * d * d + d)
    p_ln = 2 * (2 * d)
    total = vocab * d + ctx * d + n_layers * (p_attn + p_mlp + p_ln) + 2 * d
    return p_attn, total

for name, d, L, H in [("gpt2 (small)", 768, 12, 12), ("gpt2-xl", 1600, 48, 25)]:
    p_attn, total = gpt2_params(d, L, H)
    print(f"{name:13s} attention/blok {p_attn:>10,} | total {total:>13,}")


gpt2 (small)  attention/blok  2,362,368 | total   124,439,808
gpt2-xl       attention/blok 10,246,400 | total 1,557,611,200


Aritmetikanya menyusun GPT-2 dari bagian-bagian berikut:

- embedding token ($50{,}257 \times d$) dan embedding posisi ($1{,}024 \times d$);
- di setiap blok: attention dengan bias pada proyeksi query, key, dan value, karena GPT-2 asli memakai `qkv_bias=True`;
- di setiap blok: MLP yang melebarkan dimensi 4 kali lalu mengembalikannya;
- di setiap blok: dua LayerNorm, masing-masing dengan parameter skala dan geser;
- satu LayerNorm terakhir.

Lapisan output tidak dihitung terpisah, karena GPT-2 memakai ulang matriks embedding token sebagai lapisan output (*weight tying*). Hasilnya **124,439,808** parameter untuk GPT-2 small, bukan 117 juta, dan 1,557,611,200 untuk GPT-2 XL.

Cara kedua mengambil angka langsung dari checkpoint resmi di Hugging Face. Seluruh file tidak perlu diunduh: cukup header file safetensors, yang mencatat nama dan ukuran setiap tensor, ditambah satu tensor mask sebesar 4 MB, lewat permintaan HTTP yang hanya meminta potongan byte tertentu. Sel ini butuh koneksi internet.

In [50]:
import json
import math
import struct
import urllib.request

def hf_get(repo, fname, start=None, end=None):
    url = f"https://huggingface.co/{repo}/resolve/main/{fname}"
    headers = {"Range": f"bytes={start}-{end}"} if start is not None else {}
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=60) as r:
        return r.read()

hf = {}
for repo in ["openai-community/gpt2", "openai-community/gpt2-xl"]:
    cfg = json.loads(hf_get(repo, "config.json"))
    n_hdr = struct.unpack("<Q", hf_get(repo, "model.safetensors", 0, 7))[0]
    hdr = json.loads(hf_get(repo, "model.safetensors", 8, 7 + n_hdr))
    hdr.pop("__metadata__", None)
    numel = {k: math.prod(v["shape"]) for k, v in hdr.items()}
    buf = sum(n for k, n in numel.items() if k.endswith(".attn.bias"))
    hf[repo] = {"hdr": hdr, "n_hdr": n_hdr, "all": sum(numel.values()), "buf": buf}
    print(f"{repo}: n_head={cfg['n_head']} n_embd={cfg['n_embd']} n_layer={cfg['n_layer']}")
    print(f"    semua tensor {sum(numel.values()):>13,} = parameter {sum(numel.values()) - buf:>13,} + buffer mask {buf:>11,}")

g = hf["openai-community/gpt2"]
start, end = g["hdr"]["h.0.attn.bias"]["data_offsets"]
raw = hf_get("openai-community/gpt2", "model.safetensors", 8 + g["n_hdr"] + start, 8 + g["n_hdr"] + end - 1)
gpt2_mask = torch.frombuffer(bytearray(raw), dtype=torch.float32).view(1024, 1024)
print("h.0.attn.bias:", g["hdr"]["h.0.attn.bias"]["dtype"], g["hdr"]["h.0.attn.bias"]["shape"],
      "| == tril(ones):", torch.equal(gpt2_mask, torch.tril(torch.ones(1024, 1024))),
      "| == NOT mask bab ini:", torch.equal(gpt2_mask.bool(), ~torch.triu(torch.ones(1024, 1024), diagonal=1).bool()))


openai-community/gpt2: n_head=12 n_embd=768 n_layer=12
    semua tensor   137,022,720 = parameter   124,439,808 + buffer mask  12,582,912
openai-community/gpt2-xl: n_head=25 n_embd=1600 n_layer=48
    semua tensor 1,607,942,848 = parameter 1,557,611,200 + buffer mask  50,331,648
h.0.attn.bias: F32 [1, 1, 1024, 1024] | == tril(ones): True | == NOT mask bab ini: True


**GPT-2 terkecil berisi 124 juta parameter, bukan 117 juta.** Checkpoint resmi `openai-community/gpt2` memuat 124,439,808 parameter, sama persis dengan hasil aritmetika tadi. Angka 117 juta berasal dari tabel di paper GPT-2 (Radford dkk., 2019), dan muncul lagi di solusi Exercise 3.3 di repo. Buku sendiri, di Chapter 4, menamai konfigurasinya `GPT_CONFIG_124M`. Angka-angka lain di buku cocok dengan checkpoint: 12 head dan embedding 768 untuk model terkecil, serta 25 head dan embedding 1,600 untuk GPT-2 XL. Untuk ukuran XL, 1.5 miliar adalah label bulat yang lazim dipakai; hitungan dari checkpoint adalah 1,557,611,200, sekitar 1.56 miliar.

Header checkpoint juga memperlihatkan jebakan yang berkaitan langsung dengan `register_buffer` di 3.5.3. File GPT-2 small menyimpan 137,022,720 angka, bukan 124 juta, karena setiap blok membawa buffer `attn.bias` berupa mask kausal $1024 \times 1024$. Dua belas blok berarti 12,582,912 angka tambahan, kebetulan sama dengan buffer wrapper di Under the hood 4 (di sana 12 head, di sini 12 blok). Siapa pun yang menjumlahkan semua tensor di file akan mendapat 137 juta, bukan 124 juta.

Isi buffer itu adalah `tril(ones)`, dengan 1 berarti *boleh dilihat*: kebalikan persis dari mask `triu(..., diagonal=1)` di bab ini, yang memakai 1 untuk *ditutup*. Konsepnya sama, konvensinya terbalik, persis seperti perbedaan `mask_simple` dan `mask` di 3.5.1. Buffer ini juga menunjukkan bahwa checkpoint `openai-community/gpt2` di Hugging Face menyimpan mask kausal sebagai buffer konstan 4 MB di setiap blok, bukan sebagai parameter yang dilatih.

## Exercises

Solusi latihan diambil dari `exercise-solutions.ipynb` di repo. Untuk Exercise 3.1 repo menyediakan sel kode biasa; untuk Exercise 3.2 dan 3.3, solusinya hanya berupa blok kode di dalam sel markdown, dan di sini blok-blok itu dijalankan apa adanya. Semua solusi diletakkan di akhir karena solusi 3.1 mendefinisikan ulang `SelfAttention_v1`, `SelfAttention_v2`, dan `sa_v2`; kalau dijalankan di tengah bab, definisi ulang itu akan mengubah angka-angka sesudahnya. Sel versi dari notebook solusi ikut disertakan.

In [51]:
from importlib.metadata import version

import torch
print("torch version:", version("torch"))

torch version: 2.11.0+cu130


Versinya sama dengan di Setup, 2.11.0+cu130.

### Exercise 3.1 -- Comparing `SelfAttention_v1` and `SelfAttention_v2`

Soalnya: salin bobot dari sebuah `SelfAttention_v2` ke sebuah `SelfAttention_v1`, sehingga keduanya memberi output yang sama. Kuncinya ada di cara `nn.Linear` menyimpan bobotnya. Atribut `weight` berbentuk `(d_out, d_in)`, dan forward pass-nya menghitung $x W^\top$. `SelfAttention_v1` sebaliknya menyimpan matriks `(d_in, d_out)` dan menghitung $x W$. Jadi bobot yang sama harus ditransposisi saat dipindahkan. Empat sel berikut menyiapkan input, kedua kelas, dan pemindahan bobotnya.

In [52]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

d_in, d_out = 3, 2

In [53]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)

In [54]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in, d_out)

In [55]:
sa_v1.W_query = torch.nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_key = torch.nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_value = torch.nn.Parameter(sa_v2.W_value.weight.T)

Pemindahannya cukup satu `.T` pada setiap `weight`, dibungkus `nn.Parameter` baru. Ada dua detail kecil di solusi repo: kedua objek dibuat dengan seed 123, dan `SelfAttention_v2` versi solusi memakai softmax `dim=1`. Untuk tensor dua dimensi itu sama dengan `dim=-1`, seperti yang kita periksa di 3.5.1.

In [56]:
sa_v1(inputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)

Ini output `sa_v1` setelah bobotnya diganti; baris pertamanya `[-0.5337, -0.1051]`, sama dengan output yang tersimpan di repo. Baris terakhirnya, `[-0.5299, -0.1081]`, mungkin terasa familier: itu juga baris terakhir output `CausalAttention` di 3.5.3. Kedua kelas dibuat dengan seed 123 dan mengambil tiga bobot `nn.Linear(3, 2)` dalam urutan yang sama, sehingga bobotnya identik, dan token terakhir adalah satu-satunya token yang tidak tersentuh causal mask. Di baris lain, misalnya baris pertama, kedua output berbeda, karena di sana causal mask memang bekerja.

In [57]:
sa_v2(inputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)

`sa_v2` mencetak angka yang sama sampai empat desimal. Sel tambahan berikut memeriksa apakah kesamaannya sampai ke bit terakhir.

In [58]:
out_v1, out_v2 = sa_v1(inputs), sa_v2(inputs)
print("bit-identik       :", torch.equal(out_v1, out_v2))
print("max |v1 - v2|     :", (out_v1 - out_v2).abs().max().item())
print("W_query v1 == W_q.weight.T dari v2:", torch.equal(sa_v1.W_query, sa_v2.W_query.weight.T))


bit-identik       : True
max |v1 - v2|     : 0.0
W_query v1 == W_q.weight.T dari v2: True


Ya. Output kedua kelas identik bit demi bit, dan bobot hasil pemindahan identik dengan `weight.T`. Jadi, begitu bobotnya ditransposisi, kedua kelas menghitung fungsi yang sama. Yang membedakan keduanya hanyalah tata letak bobot, `(d_in, d_out)` lawan `(d_out, d_in)`, dan cara bobot itu diinisialisasi, `torch.rand` lawan inisialisasi bawaan `nn.Linear`. Kedua perbedaan itu memang disebut di buku.

### Exercise 3.2 -- Returning two-dimensional embedding vectors

Soalnya: dengan `num_heads=2` tetap, ubah *satu* argumen `MultiHeadAttentionWrapper` supaya output-nya 2 dimensi, bukan 4. Jawabannya `d_out=1`: setiap head menghasilkan satu kolom, dan dua head memberi $2 \times 1 = 2$ kolom. Sel berikut adalah blok solusi dari repo, dijalankan apa adanya.

In [59]:
torch.manual_seed(123)

d_out = 1
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]],

        [[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


Bentuknya `(2, 6, 2)`, jadi solusi repo menjawab soal dengan benar. Baris pertama batch adalah `[-0.5740, 0.2216]`.

Kejutannya ada di tempat lain. Di repo, blok ini berada di sel markdown, jadi tidak pernah dijalankan sebagai sel kode, dan di sebelahnya tercetak sebuah output yang angkanya *berbeda* dari run ini. Di Under the hood 3 kita sudah melihat bahwa angka yang berbeda belum tentu kesalahan; bisa saja hanya perbedaan RNG atau backend. Sel tambahan berikut menguji apakah selisih ini bisa dijelaskan dengan cara itu.

In [60]:
repo_claim = torch.tensor([[-9.1476e-02,  3.4164e-02],
                           [-2.6796e-01, -1.3427e-03],
                           [-4.8421e-01, -4.8909e-02],
                           [-6.4808e-01, -1.0625e-01],
                           [-8.8380e-01, -1.7140e-01],
                           [-1.4744e+00, -3.4327e-01]])
live = context_vecs[0].detach()
x_l1 = inputs.abs().sum(dim=1)
bound = x_l1.max().item() / d_in**0.5
w_max = max(h.W_value.weight.abs().max().item() for h in mha.heads)
print("live == klaim repo (atol 1e-4)?   :", torch.allclose(live, repo_claim, atol=1e-4))
print(f"max |live - klaim repo|           : {(live - repo_claim).abs().max().item():.4f}")
print(f"init nn.Linear: |w| <= 1/sqrt(d_in) = {1 / d_in**0.5:.4f}; max |W_value| run ini = {w_max:.4f}")
print(f"max_t ||x_t||_1 = {x_l1.max().item():.2f} -> batas |z| untuk SEED APA PUN = {bound:.4f}")
print(f"|z| terbesar: live {live.abs().max().item():.4f} | klaim repo {repo_claim.abs().max().item():.4f}")


live == klaim repo (atol 1e-4)?   : False
max |live - klaim repo|           : 0.8320
init nn.Linear: |w| <= 1/sqrt(d_in) = 0.5774; max |W_value| run ini = 0.4590
max_t ||x_t||_1 = 2.08 -> batas |z| untuk SEED APA PUN = 1.2009
|z| terbesar: live 0.7774 | klaim repo 1.4744


**Output yang tercetak di solusi Exercise 3.2 di repo tidak mungkin dihasilkan oleh kode solusinya sendiri.** Selisih terbesarnya dengan run ini 0.8320, terlalu besar untuk pembulatan, dan ada dua alasan kenapa penyebabnya juga bukan RNG atau backend.

Alasan pertama berlaku untuk seed apa pun. Dengan dropout 0, output setiap head adalah rata-rata berbobot dari value-nya, dengan bobot positif yang berjumlah 1, sehingga nilai mutlaknya tidak mungkin melampaui value terbesar: $|z_t| \le \max_s |v_s|$. Dengan `d_out=1`, setiap value hanya satu angka, $v_s = x_s \cdot w$. Inisialisasi bawaan `nn.Linear` menjamin setiap $|w_i| \le 1/\sqrt{3} = 0.5774$ (di run ini yang terbesar 0.4590), sehingga $|v_s| \le \|x_s\|_1 / \sqrt{3}$. Embedding dengan jumlah nilai mutlak terbesar adalah "journey", $0.55 + 0.87 + 0.66 = 2.08$, jadi untuk seed apa pun $|z_t| \le 2.08 / \sqrt{3} = 1.2009$. Output di repo memuat 1.4744 di baris terakhirnya, di atas batas itu.

Alasan kedua khusus untuk lingkungan run ini. Generator acak di sini menghasilkan output wrapper dengan `d_out=2` di 3.6.1 yang sama dengan yang dicetak buku. Dengan generator yang sama, run `d_out=1` memberi nilai mutlak terbesar 0.7774, jauh dari 1.4744.

Jadi kode solusinya benar; yang keliru adalah output yang tercetak di sebelahnya, kemungkinan besar sisa dari versi kode yang lain, walaupun penyebab pastinya tidak bisa ditelusuri dari repo. Bagi pembaca, artinya sederhana: kode solusi itu bisa dipakai apa adanya, dan angka yang benar-benar dihasilkannya adalah angka yang dicetak sel solusi di atas, bukan angka yang tercetak di repo.

### Exercise 3.3 -- Initializing GPT-2 size attention modules

Soalnya: buat sebuah `MultiHeadAttention` seukuran GPT-2 small, yaitu 12 head, `d_in = d_out = 768`, dan konteks 1,024 token.

In [61]:
context_length = 1024
d_in, d_out = 768, 768
num_heads = 12

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads)

Modulnya terbentuk tanpa output. Lebar setiap head adalah $768 / 12 = 64$, jadi skor di dalamnya dibagi $\sqrt{64} = 8$, bukan $\sqrt{768}$, nuansa yang sudah kita catat di Under the hood 1. Blok kedua solusi menghitung jumlah parameternya.

In [62]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

count_parameters(mha)

2360064

Hasilnya 2,360,064 parameter, sama dengan angka di solusi repo (`2360064  # (2.36 M)`) dan dengan `MultiHeadAttention` ukuran GPT-2 di Under the hood 4. Rinciannya: $3 \times 768^2 = 1{,}769{,}472$ untuk proyeksi query, key, dan value tanpa bias, ditambah `out_proj` sebesar $768^2 + 768 = 590{,}592$. Buffer mask $1024 \times 1024$ tidak ikut terhitung, karena `count_parameters` hanya menjumlahkan `parameters()`.

Solusi repo ditutup dengan dua pernyataan: GPT-2 berisi 117 juta parameter, dan sebagian besar parameternya tidak berada di attention. Angka totalnya sudah kita koreksi di atas. Pernyataan kedua bisa diperiksa dengan aritmetika yang sama, termasuk bias query, key, dan value seperti GPT-2 asli:

| Bagian GPT-2 small | Parameter | Porsi |
|---|---|---|
| embedding token + posisi | 39,383,808 | 31.6% |
| attention, 12 blok | 28,348,416 | 22.8% |
| MLP, 12 blok | 56,669,184 | 45.5% |
| LayerNorm | 38,400 | 0.03% |
| **total** | **124,439,808** | 100% |

Pernyataan itu tepat. Attention memegang kurang dari seperempat parameter, sedangkan hampir separuhnya berada di MLP dan hampir sepertiganya di embedding. Modul yang menjadi pusat bab ini ternyata bukan bagian yang paling banyak memakan parameter.

## Summary

Bab ini membangun attention dari nol dalam empat tahap, dan setiap tahap menambal kekurangan tahap sebelumnya. Semuanya berangkat dari masalah di 3.1: encoder-decoder RNN harus memadatkan seluruh kalimat ke satu hidden state. Attention membiarkan setiap posisi menengok langsung ke semua posisi lain, dan self-attention menerapkan gagasan itu di dalam satu sekuens.

Self-attention sederhana memperlihatkan inti mekanismenya: skor dari dot product, bobot dari softmax, dan context vector sebagai rata-rata berbobot. Untuk "journey", hasilnya $z^{(2)} =$ `[0.4419, 0.6515, 0.5683]`, tertarik ke arah "journey" dan "starts" yang memegang hampir setengah bobot. Tapi dot product mentah ikut dipengaruhi panjang vektor, sampai-sampai "starts" memberi skor lebih tinggi kepada "journey", vektor terpanjang di kalimat itu, daripada kepada dirinya sendiri. Di bagian yang sama kita juga melihat kenapa softmax naif tidak dipakai: input sekitar 1,000 sudah membuatnya `nan`, dan mengurangi nilai maksimum lebih dulu menyelesaikan masalah itu.

Matriks $W_q$, $W_k$, dan $W_v$ memberi model kebebasan untuk mempelajari sendiri apa yang relevan, dan skornya dibagi $\sqrt{d_k}$ sebelum softmax. Under the hood 1 memperlihatkan alasannya: tanpa pembagi, di $d_k = 4096$ bobot terbesar rata-rata 0.986 dan softmax nyaris berhenti bereaksi, sedangkan dengan pembagi softmax bekerja di daerah yang sama untuk semua ukuran.

Causal mask mengisi skor masa depan dengan $-\infty$ sebelum softmax. Under the hood 2 mengujinya dari dua arah: mengganggu token 4-6 tidak mengubah satu bit pun output token 1-3, dan semua turunan terhadap token masa depan tepat nol. Dropout menambahkan keacakan selama training, dengan akibat yang mudah terlewat: di contoh buku seluruh bobot "journey" terkena dropout sehingga context vector-nya nol, jumlah satu baris bisa 0 atau 2, dan hanya `eval()`, bukan `no_grad()`, yang mematikan dropout. Mask dropout dari CPU sama dengan buku, sedangkan mask dari GPU berbeda, dan perbedaan seperti itu adalah soal RNG atau backend, bukan bug.

`CausalAttention` membungkus semua itu untuk input berbatch, dengan mask yang disimpan sebagai buffer. Karena ukuran buffer itu tetap, `context_length` menjadi batas keras: tujuh token untuk modul yang dibuat untuk enam berhenti dengan `RuntimeError`.

Multi-head attention menjalankan beberapa head berdampingan. `MultiHeadAttention` memecah satu proyeksi besar menjadi beberapa head dengan `.view` dan `.transpose`, sehingga di kelas ini `d_out` berarti lebar total, bukan lebar per head. Versi wrapper dan versi weight split menghitung fungsi yang sama, dengan selisih hanya 5.96e-08 setelah transplantasi bobot; `MultiHeadAttention` menyimpan satu mask, bukan dua belas, tapi menambahkan `out_proj`. Keunggulan efisiensi yang disebut buku untuk versi weight split tidak terlihat di CPU satu thread pada run ini, dan yang jelas lebih cepat adalah fungsi bawaan PyTorch, SDPA.

Bagian latihan dan dua pemeriksaan menutup bab. Exercise 3.1 menunjukkan bahwa `SelfAttention_v1` dan `SelfAttention_v2` memberi output identik bit demi bit begitu bobot `nn.Linear` disalin dengan transposisi. GPT-2 terkecil berisi 124,439,808 parameter, bukan 117 juta, dan attention hanya memegang kurang dari seperempatnya. Output yang tercetak di solusi Exercise 3.2 di repo melampaui batas yang berlaku untuk seed apa pun, sehingga bukan perbedaan RNG, melainkan output yang keliru.

## What comes next

Chapter 4 memasang `MultiHeadAttention` dari bab ini ke dalam *transformer block*, bersama LayerNorm, MLP dengan aktivasi GELU, dan shortcut connection. Blok-blok itu ditumpuk menjadi GPT seukuran `GPT_CONFIG_124M`, lalu jumlah parameternya dihitung lagi. Hitungan di sana bisa dicocokkan dengan aritmetika di akhir 3.6.2. Dua pengaturan menentukan apakah angkanya sama: apakah lapisan output berbagi bobot dengan embedding token (*weight tying*), dan apakah proyeksi query, key, dan value memakai bias (`qkv_bias`).

Semua output di notebook ini berasal dari satu run di Linux x86_64 dengan GPU CUDA (Python 3.13.15, PyTorch 2.11.0+cu130). Perhitungan bab sendiri berjalan di CPU; GPU hanya dipakai untuk perbandingan mask dropout di Under the hood 3.